In [3]:
"""
Causal SHAP (paper-faithful) for CSVs with auto regression/classification
- Tested for:
  /mnt/data/cv_risk.csv          (continuous target: cv_risk)
  /mnt/data/lung_cancer_risk.csv (continuous target: lung_cancer_risk)
"""

import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
from typing import Dict, List, Set, Optional
warnings.filterwarnings("ignore")

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import f1_score, accuracy_score, r2_score, mean_absolute_error
from sklearn.model_selection import train_test_split
from cgt_xai.causal_shap_robustness import run_robustness_analysis
from sklearn.utils.multiclass import type_of_target
import networkx as nx

try:
    from xgboost import XGBClassifier, XGBRegressor
    HAVE_XGB = True
except Exception:
    HAVE_XGB = False

HAVE_XGB = False

from causallearn.search.ConstraintBased.PC import pc

CONFIG = {
    'ALPHA_PC': 0.1,
    'MAX_PERM': 200,
    'M_RESAMPLES': 64,
    'GLOBAL_K': 30,
    'RANDOM_SEED': 42,
    'N_DAGS': 5
}
np.random.seed(CONFIG['RANDOM_SEED'])
rng = np.random.default_rng(CONFIG['RANDOM_SEED'])

COMMON_TARGET_NAMES = ["target","label","y","class","outcome",
                       "cvd","cardio","TenYearCHD","heart_disease","stroke","risk",
                       "cancer","lung_cancer","has_cancer","diagnosis"]

def detect_target_column(df: pd.DataFrame, user_hint: Optional[str]=None) -> str:
    if user_hint and user_hint in df.columns: return user_hint
    lower_map = {c.lower(): c for c in df.columns}
    for name in COMMON_TARGET_NAMES:
        if name.lower() in lower_map: return lower_map[name.lower()]
    return df.columns[-1]

def drop_unnamed(df: pd.DataFrame) -> pd.DataFrame:
    return df[[c for c in df.columns if not re.match(r'^unnamed', c, re.I)]].copy()

def to_numeric_df(df: pd.DataFrame) -> pd.DataFrame:
    num = df.select_dtypes(include=[np.number]).copy()
    for c in df.columns:
        if c not in num.columns:
            try:
                num[c] = pd.to_numeric(df[c], errors='raise')
            except Exception:
                pass
    return num

class CausalStructure:
    def __init__(self, alpha=0.05):
        self.alpha = alpha
        self.cpdag = None
        self.feature_names: List[str] = []
        self.dags: List[nx.DiGraph] = []

    def discover(self, X: pd.DataFrame):
        self.feature_names = list(X.columns)
        print(f"Running PC on {len(self.feature_names)} features...")
        res = pc(X.values, alpha=self.alpha, show_progress=False)
        self.cpdag = res.G.graph
        print("PC completed -> CPDAG")
        return self

    def get_skeleton(self) -> nx.DiGraph:
        G = nx.DiGraph(); G.add_nodes_from(self.feature_names)
        n = len(self.feature_names)
        for i in range(n):
            for j in range(n):
                if i == j: continue
                if self.cpdag[i, j] == -1 and self.cpdag[j, i] == 1:
                    G.add_edge(self.feature_names[i], self.feature_names[j])
                elif self.cpdag[i, j] == -1 and self.cpdag[j, i] == -1 and i < j:
                    G.add_edge(self.feature_names[i], self.feature_names[j])
                    G.add_edge(self.feature_names[j], self.feature_names[i])
        return G

    def sample_dags(self, n_samples: int) -> List[nx.DiGraph]:
        dags = []
        names = self.feature_names; n = len(names)
        undirected = []
        for i in range(n):
            for j in range(i+1, n):
                if self.cpdag[i, j] == -1 and self.cpdag[j, i] == -1:
                    undirected.append((names[i], names[j]))
        for _ in range(max(1, n_samples)):
            G = nx.DiGraph(); G.add_nodes_from(names)
            for i in range(n):
                for j in range(n):
                    if i != j and self.cpdag[i, j] == -1 and self.cpdag[j, i] == 1:
                        G.add_edge(names[i], names[j])
            for u, v in undirected:
                if rng.random() < 0.5: G.add_edge(u, v)
                else: G.add_edge(v, u)
                if not nx.is_directed_acyclic_graph(G):
                    if G.has_edge(u, v): G.remove_edge(u, v); G.add_edge(v, u)
                    else: G.remove_edge(v, u); G.add_edge(u, v)
                if not nx.is_directed_acyclic_graph(G):
                    if G.has_edge(u, v): G.remove_edge(u, v)
                    if G.has_edge(v, u): G.remove_edge(v, u)
            if not nx.is_directed_acyclic_graph(G):
                for (a, b) in list(G.edges()):
                    G.remove_edge(a, b)
                    if nx.is_directed_acyclic_graph(G): break
                    else: G.add_edge(a, b)
            dags.append(G)
        self.dags = dags
        print(f"Sampled {len(dags)} DAG(s)")
        return dags

class LinearMechanisms:
    def __init__(self, dag: nx.DiGraph, feature_names: List[str]):
        self.dag = dag; self.feature_names = feature_names
        self.mechanisms: Dict[str, Dict] = {}; self.topo: List[str] = []

    def fit(self, X: pd.DataFrame):
        try: self.topo = list(nx.topological_sort(self.dag))
        except Exception: self.topo = self._safe_topo()
        for var in self.feature_names:
            parents = list(self.dag.predecessors(var))
            if not parents:
                self.mechanisms[var] = {"parents": [], "intercept": float(X[var].mean()),
                                        "coef": None, "sigma": float(X[var].std(ddof=0))}
            else:
                lr = LinearRegression().fit(X[parents].values, X[var].values)
                resid = X[var].values - lr.predict(X[parents].values)
                self.mechanisms[var] = {"parents": parents, "intercept": float(lr.intercept_),
                                        "coef": lr.coef_.astype(float), "sigma": float(np.std(resid, ddof=0))}
        return self

    def _safe_topo(self) -> List[str]:
        ordered, remaining = [], set(self.feature_names)
        while remaining:
            ready = [n for n in remaining if all(p in ordered or p not in remaining
                                                 for p in self.dag.predecessors(n))]
            if not ready:
                ready = [min(remaining, key=lambda n: len([p for p in self.dag.predecessors(n) if p in remaining]))]
            for n in ready: ordered.append(n); remaining.remove(n)
        return ordered

    def sample_intervention(self, S: Set[str], x_obs: pd.Series) -> Dict[str, float]:
        x_new: Dict[str, float] = {}
        for var in self.topo:
            if var in S: x_new[var] = float(x_obs[var]); continue
            mech = self.mechanisms[var]
            if not mech["parents"]:
                mu, sigma = mech["intercept"], mech["sigma"]
                x_new[var] = mu if sigma < 1e-8 else float(rng.normal(mu, sigma))
            else:
                pvals = np.array([x_new.get(p, x_obs[p]) for p in mech["parents"]])
                mu = mech["intercept"] + pvals @ mech["coef"]
                sigma = mech["sigma"]
                x_new[var] = mu if sigma < 1e-8 else float(rng.normal(mu, sigma))
        return x_new

class CausalEffects:
    def __init__(self, dags: List[nx.DiGraph], feature_names: List[str]):
        self.dags = dags
        self.feature_names = feature_names
        self.effects: Dict[str, float] = {}

    def compute_effects(self, X: pd.DataFrame, y: pd.Series) -> Dict[str, float]:
        print("Computing IDA-style total effects W_i ...")
        y_vals = y.values
        totals = {f: [] for f in self.feature_names}

        for dag in self.dags:
            G = dag.copy()
            G.add_node("Y")

            # (A) Edge weights among features
            for u, v in dag.edges():
                parents = [p for p in dag.predecessors(v) if p != u]
                X_reg = np.column_stack([X[parents].values, X[u].values]) if parents else X[[u]].values
                lr = LinearRegression().fit(X_reg, X[v].values)
                w = float(abs(lr.coef_[-1]))
                if w > 1e-12:
                    G.add_edge(u, v, weight=w)

            # (B) Direct edges into Y: Y ~ all features
            paY = list(self.feature_names)
            lrY = LinearRegression().fit(X[paY].values, y_vals)
            for feat, w in zip(paY, np.abs(lrY.coef_)):
                if w > 1e-12:
                    G.add_edge(feat, "Y", weight=float(w))

            # (C) Sum of path products i -> ... -> Y
            for i_feat in self.feature_names:
                total = 0.0
                try:
                    for path in nx.all_simple_paths(G, i_feat, "Y", cutoff=10):
                        prod = 1.0
                        ok = True
                        for a, b in zip(path[:-1], path[1:]):
                            ed = G.get_edge_data(a, b)
                            if not ed or "weight" not in ed:
                                ok = False
                                break
                            prod *= ed["weight"]
                        if ok:
                            total += prod
                except Exception:
                    pass
                totals[i_feat].append(total)

        self.effects = {f: (float(np.mean(vals)) if len(vals) else 0.0) for f, vals in totals.items()}
        print("W_i computed")
        return self.effects

    def gamma_weights(self) -> Dict[str, float]:
        absW = {k: abs(v) for k, v in self.effects.items()}
        Z = sum(absW.values()) + 1e-12
        return {k: v / Z for k, v in absW.items()}


class CausalSHAP:
    def __init__(self, model, structure: CausalStructure, X_train: pd.DataFrame, y_train: pd.Series, task: str):
        self.model = model; self.structure = structure
        self.X_train = X_train; self.y_train = y_train
        self.features = list(X_train.columns); self.task = task

        print("\nFitting SEMs & sampling DAGs ...")
        self.dags = self.structure.sample_dags(CONFIG['N_DAGS'])
        self.sems = [LinearMechanisms(d, self.features).fit(X_train) for d in self.dags]

        self.effects = CausalEffects(self.dags, self.features)
        self.effects.compute_effects(X_train, y_train)
        self.gamma = self.effects.gamma_weights()
        self._Ef_cache = None
        print("CausalSHAP ready")

    def _predict_scalar(self, X_mat: np.ndarray) -> float:
        if self.task == "classification":
            proba = self.model.predict_proba(X_mat)[0]
            return float(proba[np.argmax(proba)])  # predicted class prob (paper used f(x) scalar)
        else:  # regression
            return float(self.model.predict(X_mat)[0])

    def _Ef(self) -> float:
        if self._Ef_cache is None:
            if self.task == "classification":
                P = self.model.predict_proba(self.X_train.values)
                self._Ef_cache = float(P.max(axis=1).mean())  # mean predicted-class prob
            else:
                preds = self.model.predict(self.X_train.values)
                self._Ef_cache = float(np.mean(preds))
        return self._Ef_cache

    def _vc(self, S: Set[str], x: pd.Series, M: int) -> float:
        total = 0.0
        per = max(1, M // max(1, len(self.sems)))
        for sem in self.sems:
            for _ in range(per):
                x_sample = sem.sample_intervention(S, x)
                X_mat = pd.DataFrame([x_sample], columns=self.features).values
                total += self._predict_scalar(X_mat)
        N = per * max(1, len(self.sems))
        return total / N if N > 0 else 0.0

    def explain_instance(self, x: pd.Series, n_perms: int=None, M: int=None) -> pd.Series:
        if n_perms is None: n_perms = CONFIG['MAX_PERM']
        if M is None: M = CONFIG['M_RESAMPLES']

        fx = self._predict_scalar(pd.DataFrame([x], columns=self.features).values)
        phi = np.zeros(len(self.features), dtype=float)

        for _ in range(n_perms):
            perm = rng.permutation(self.features)
            S: Set[str] = set()
            v_prev = self._vc(S, x, M)
            for feat in perm:
                S_next = S | {feat}
                v_curr = self._vc(S_next, x, M)
                delta = v_curr - v_prev
                phi[self.features.index(feat)] += self.gamma[feat] * delta
                v_prev, S = v_curr, S_next

        phi /= n_perms

        sum_phi_c = float(np.sum(phi))
        if abs(sum_phi_c) > 1e-12:
            Ef = self._Ef()
            phi = phi * ((fx - Ef) / sum_phi_c)
        return pd.Series(phi, index=self.features, name="Causal_SHAP")

    def explain_global(self, X_test: pd.DataFrame, k: int=None) -> pd.Series:
        if k is None: k = CONFIG['GLOBAL_K']
        k = min(k, len(X_test))
        idx = rng.choice(len(X_test), size=k, replace=False)
        print(f"\nComputing global Causal SHAP over {k} instances ...")
        exps = []
        for t, i in enumerate(idx, 1):
            if t % 5 == 0: print(f"  {t}/{k}")
            exps.append(self.explain_instance(X_test.iloc[i], n_perms=max(10, CONFIG['MAX_PERM']//4)))
        return pd.concat(exps, axis=1).mean(axis=1).rename("Causal_SHAP_Global")

def annotate_heatmap(ax, data: pd.DataFrame, title: str, cbar_label: str):
    im = ax.imshow(data.values, aspect='auto')
    ax.set_xticks(np.arange(data.shape[1])); ax.set_xticklabels(data.columns, rotation=45, ha='right')
    ax.set_yticks(np.arange(data.shape[0])); ax.set_yticklabels(data.index)
    for i in range(data.shape[0]):
        for j in range(data.shape[1]):
            ax.text(j, i, f"{data.values[i, j]:.3f}", ha='center', va='center', fontsize=8)
    cbar = plt.colorbar(im, ax=ax); cbar.set_label(cbar_label)
    ax.set_title(title)

def plot_comparison(df: pd.DataFrame, title="Feature Attribution Comparison"):
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    axes[0, 0].bar(np.arange(df.shape[0]), df.iloc[:, 0], width=0.6)
    axes[0, 0].set_title(f'{title}\n(Bar: first column)', fontsize=12, fontweight='bold')
    axes[0, 0].set_xticks(np.arange(df.shape[0])); axes[0, 0].set_xticklabels(df.index, rotation=45, ha='right')
    axes[0, 0].set_ylabel('Attribution'); axes[0, 0].grid(axis='y', alpha=0.3)
    annotate_heatmap(axes[0, 1], df.T, 'Attribution Heatmap', 'Score')
    df_norm = df.div(df.abs().max(axis=0), axis=1)
    for i, col in enumerate(df_norm.columns):
        axes[1, 0].bar(np.arange(df_norm.shape[0]) + (i-0.5)*(0.8/len(df_norm.columns)), df_norm[col],
                       width=0.8/len(df_norm.columns), label=col)
    axes[1, 0].set_title('Normalized by method max'); axes[1, 0].legend(fontsize=9); axes[1, 0].grid(axis='y', alpha=0.3)
    ranks = df.rank(ascending=False)
    annotate_heatmap(axes[1, 1], ranks.T, 'Ranking (lower is better)', 'Rank')
    fig.tight_layout(); return fig

def plot_causal_graph(G: nx.DiGraph, title="Discovered Causal Graph (PC)"):
    plt.figure(figsize=(8, 6))
    pos = nx.spring_layout(G, seed=CONFIG['RANDOM_SEED'], k=1.0, iterations=200)
    nx.draw_networkx_nodes(G, pos, node_color='lightblue', node_size=1200, alpha=0.9)
    nx.draw_networkx_edges(G, pos, edge_color='gray', arrows=True, arrowsize=15, width=2, arrowstyle='->', connectionstyle='arc3,rad=0.1')
    nx.draw_networkx_labels(G, pos, font_size=10, font_weight='bold')
    plt.title(title, fontsize=13, fontweight='bold', pad=12)
    plt.axis('off'); plt.tight_layout()
    return plt.gcf()

def run_causal_shap_on_csv(csv_path: str, dataset_name: Optional[str]=None, target_hint: Optional[str]=None, test_size: float=0.25):
    print("\n" + "="*72)
    print(f"Causal SHAP (paper-faithful) on: {csv_path}")
    print("="*72)

    try:
        raw = pd.read_csv(csv_path)
    except Exception as e:
        print(f"ERROR reading CSV: {e}"); return

    raw = drop_unnamed(raw)  # drop index columns like 'Unnamed: 0'
    if dataset_name is None:
        dataset_name = csv_path.split("/")[-1].split(".")[0]

    tgt_col = detect_target_column(raw, target_hint)
    if tgt_col not in raw.columns:
        print("ERROR: target column not found."); return

    y_raw = raw[tgt_col].copy()
    X_raw = raw.drop(columns=[tgt_col])

    X_num = to_numeric_df(X_raw)
    dropped = set(X_raw.columns) - set(X_num.columns)
    if dropped:
        print(f"Note: Dropping non-numeric columns: {sorted(list(dropped))}")

    X_num = X_num.replace([np.inf, -np.inf], np.nan).fillna(X_num.median(numeric_only=True))

    y_type = type_of_target(y_raw)
    if y_type in ["binary","multiclass"]:
        task = "classification"
        y_vec = y_raw.astype(int).values
    else:
        task = "regression"
        y_vec = pd.to_numeric(y_raw, errors="coerce").values
        if np.isnan(y_vec).any():
            print("ERROR: target has non-numeric values that cannot be coerced."); return

    strat = y_vec if (task=="classification" and len(np.unique(y_vec))>1) else None
    X_train, X_test, y_train, y_test = train_test_split(
        X_num, y_vec, test_size=test_size, random_state=CONFIG['RANDOM_SEED'], stratify=strat
    )

    scaler = StandardScaler()
    Xtr = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
    Xte = pd.DataFrame(scaler.transform(X_test),  columns=X_test.columns,  index=X_test.index)

    if task=="classification":
        n_classes = int(len(np.unique(y_train)))
        if HAVE_XGB:
            model = XGBClassifier(
                n_estimators=160, max_depth=5, learning_rate=0.08,
                subsample=0.9, colsample_bytree=0.9,
                random_state=CONFIG['RANDOM_SEED'], verbosity=0,
                objective="multi:softprob" if n_classes>2 else "binary:logistic",
                num_class=n_classes if n_classes>2 else None
            )
            print("Using XGBoost (classification)")
        else:
            model = RandomForestClassifier(n_estimators=160, max_depth=6, random_state=CONFIG['RANDOM_SEED'])
            print("Using RandomForest (classification)")
        model.fit(Xtr.values, y_train)
        y_pred = model.predict(Xte.values)
        acc = accuracy_score(y_test, y_pred); f1m = f1_score(y_test, y_pred, average='macro')
        print(f"Model: acc={acc:.4f}, macroF1={f1m:.4f}, classes={n_classes}")
    else:
        if HAVE_XGB:
            model = XGBRegressor(
                n_estimators=200, max_depth=5, learning_rate=0.08,
                subsample=0.9, colsample_bytree=0.9,
                random_state=CONFIG['RANDOM_SEED']
            )
            print("Using XGBoost (regression)")
        else:
            model = RandomForestRegressor(n_estimators=200, max_depth=6, random_state=CONFIG['RANDOM_SEED'])
            print("Using RandomForest (regression)")
        model.fit(Xtr.values, y_train)
        y_pred = model.predict(Xte.values)
        r2 = r2_score(y_test, y_pred); mae = mean_absolute_error(y_test, y_pred)
        print(f"Model: R2={r2:.4f}, MAE={mae:.4f}")

    # Step 1: PC discovery
    print("\n" + "-"*64)
    print("Step 1: Causal discovery (PC)")
    print("-"*64)
    cs = CausalStructure(alpha=CONFIG['ALPHA_PC']).discover(Xtr)
    G_skel = cs.get_skeleton()
    print(f"Graph: {G_skel.number_of_nodes()} nodes, {G_skel.number_of_edges()} edges")
    fig = plot_causal_graph(G_skel, title=f"Discovered Causal Graph (PC) — {dataset_name}")
    fig.savefig(f'causal_graph_{dataset_name}.png', dpi=150, bbox_inches='tight')
    print(f"Saved 'causal_graph_{dataset_name}.png'")

    # Step 2: Causal SHAP
    print("\n" + "-"*64)
    print("Step 2: Causal SHAP")
    print("-"*64)
    cshap = CausalSHAP(model, cs, Xtr, pd.Series(y_train), task=task)
    G_aug = make_augmented_graph_avg(cshap.dags, Xtr, pd.Series(y_train))
    fig_aug = plot_causal_graph_with_weights(G_aug, title=f"Augmented Causal Graph with target — {dataset_name}")
    fig_aug.savefig(f"causal_graph_with_target_{dataset_name}.png", dpi=150, bbox_inches="tight")
    print(f"Saved 'causal_graph_with_target_{dataset_name}.png'")
    print("\nTotal causal effects W_i and gamma_i:")
    for f in Xtr.columns:
        print(f"  {f:20s}  W={cshap.effects.effects[f]:10.6f}   γ={cshap.gamma[f]:.6f}")

    glob = cshap.explain_global(Xte, k=min(CONFIG['GLOBAL_K'], len(Xte)))

    # Step 3: Save/plot comparison (Causal_SHAP only by default for regression)
    cols = {"Causal_SHAP": glob}
    df = pd.DataFrame(cols)
    df.to_csv(f"attribution_scores_{dataset_name}.csv", index=True)
    print("\n" + "-"*64)
    print("Step 3: Comparison & Save")
    print("-"*64)
    print(df.round(6).to_string())
    print(f"\nSaved: attribution_scores_{dataset_name}.csv")

    fig2 = plot_comparison(df, title=f"Feature Attribution Comparison — {dataset_name}")
    fig2.savefig(f'attribution_comparison_{dataset_name}.png', dpi=150, bbox_inches='tight')
    print(f"Saved 'attribution_comparison_{dataset_name}.png'")

    try: plt.show()
    except Exception: pass


def make_augmented_graph_avg(dags: List[nx.DiGraph], X: pd.DataFrame, y: pd.Series) -> nx.DiGraph:
    """Average the feature→feature and feature→Y weights across sampled DAGs, then return a single augmented graph."""
    from collections import defaultdict

    edge_weights = defaultdict(list)
    nodes = set(X.columns.tolist() + ["Y"])

    for dag in dags:
        G = dag.copy()
        G.add_node("Y")

        # (A) feature->feature weights (parent-conditioned)
        for u, v in dag.edges():
            parents = [p for p in dag.predecessors(v) if p != u]
            X_reg = np.column_stack([X[parents].values, X[u].values]) if parents else X[[u]].values
            lr = LinearRegression().fit(X_reg, X[v].values)
            w = float(abs(lr.coef_[-1]))
            if w > 1e-12:
                edge_weights[(u, v)].append(w)

        # (B) feature->Y weights (Y ~ all features)
        lrY = LinearRegression().fit(X.values, y.values)
        for feat, w in zip(X.columns, np.abs(lrY.coef_)):
            if w > 1e-12:
                edge_weights[(feat, "Y")].append(float(w))

    G_avg = nx.DiGraph()
    G_avg.add_nodes_from(nodes)
    for (u, v), ws in edge_weights.items():
        G_avg.add_edge(u, v, weight=float(np.mean(ws)))
    return G_avg

def plot_causal_graph_with_weights(G: nx.DiGraph, title: str):
    import matplotlib.pyplot as plt
    pos = nx.spring_layout(G, seed=CONFIG['RANDOM_SEED'], k=1.0, iterations=200)
    plt.figure(figsize=(9, 7))
    nx.draw_networkx_nodes(G, pos, node_color='lightblue', node_size=1200, alpha=0.9)
    nx.draw_networkx_edges(G, pos, edge_color='gray', arrows=True, arrowsize=15,
                           width=2, arrowstyle='->', connectionstyle='arc3,rad=0.1')
    nx.draw_networkx_labels(G, pos, font_size=10, font_weight='bold')
    # edge labels = average weights
    edge_lbls = {(u, v): f"{d.get('weight', 0):.2f}" for u, v, d in G.edges(data=True)}
    nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_lbls, font_size=8)
    plt.title(title, fontsize=13, fontweight='bold', pad=12)
    plt.axis('off'); plt.tight_layout()
    return plt.gcf()

def make_augmented_graph_avg(dags: List[nx.DiGraph], X: pd.DataFrame, y: pd.Series) -> nx.DiGraph:
    """Average the feature→feature and feature→Y weights across sampled DAGs, then return a single augmented graph."""
    from collections import defaultdict

    edge_weights = defaultdict(list)
    nodes = set(X.columns.tolist() + ["Y"])

    for dag in dags:
        G = dag.copy()
        G.add_node("Y")

        # (A) feature->feature weights (parent-conditioned)
        for u, v in dag.edges():
            parents = [p for p in dag.predecessors(v) if p != u]
            X_reg = np.column_stack([X[parents].values, X[u].values]) if parents else X[[u]].values
            lr = LinearRegression().fit(X_reg, X[v].values)
            w = float(abs(lr.coef_[-1]))
            if w > 1e-12:
                edge_weights[(u, v)].append(w)

        # (B) feature->Y weights (Y ~ all features)
        lrY = LinearRegression().fit(X.values, y.values)
        for feat, w in zip(X.columns, np.abs(lrY.coef_)):
            if w > 1e-12:
                edge_weights[(feat, "Y")].append(float(w))

    G_avg = nx.DiGraph()
    G_avg.add_nodes_from(nodes)
    for (u, v), ws in edge_weights.items():
        G_avg.add_edge(u, v, weight=float(np.mean(ws)))
    return G_avg

def plot_causal_graph_with_weights(G: nx.DiGraph, title: str):
    import matplotlib.pyplot as plt
    pos = nx.spring_layout(G, seed=CONFIG['RANDOM_SEED'], k=1.0, iterations=200)
    plt.figure(figsize=(9, 7))
    nx.draw_networkx_nodes(G, pos, node_color='lightblue', node_size=1200, alpha=0.9)
    nx.draw_networkx_edges(G, pos, edge_color='gray', arrows=True, arrowsize=15,
                           width=2, arrowstyle='->', connectionstyle='arc3,rad=0.1')
    nx.draw_networkx_labels(G, pos, font_size=10, font_weight='bold')
    # edge labels = average weights
    edge_lbls = {(u, v): f"{d.get('weight', 0):.2f}" for u, v, d in G.edges(data=True)}
    nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_lbls, font_size=8)
    plt.title(title, fontsize=13, fontweight='bold', pad=12)
    plt.axis('off'); plt.tight_layout()
    return plt.gcf()

def main():
    run_causal_shap_on_csv("cv_risk.csv", dataset_name="cv_risk")
    run_causal_shap_on_csv("lung_cancer_risk.csv", dataset_name="lung_cancer_risk")

    run_robustness_analysis([
        {"path": "../../datasets/cv_risk.csv",         "name": "cv_risk"},
        {"path": "../../datasets/lung_cancer_risk.csv", "name": "lung_cancer_risk"},
    ])

if __name__ == "__main__":
    main()


Causal SHAP (paper-faithful) on: cv_risk.csv
ERROR reading CSV: [Errno 2] No such file or directory: 'cv_risk.csv'

Causal SHAP (paper-faithful) on: lung_cancer_risk.csv
ERROR reading CSV: [Errno 2] No such file or directory: 'lung_cancer_risk.csv'

  Robustness analysis: cv_risk  (datasets/cv_risk.csv)
Task: regression  |  4 features  |  750 train / 250 test samples
✓ Model trained
Running PC causal discovery ...
✓ Sampled 5 DAGs
✓ SEMs fitted
✓ γ weights computed

Computing baseline global SHAP (k=15) ...
Baseline φ₀:
  bmi                        -0.315155
  sleep_duration             -0.033617
  diet_score                 +0.029888
  family_history             +0.003239

Running robustness sweep over ε=[0.01, 0.05, 0.1, 0.2, 0.3, 0.5, 1.0] (5 trials each) ...
  ε=0.010 ...  MAD=0.23052  ρ=0.7600
  ε=0.050 ...  MAD=0.17774  ρ=0.6800
  ε=0.100 ...  MAD=0.20850  ρ=0.8800
  ε=0.200 ...  MAD=0.48671  ρ=0.9200
  ε=0.300 ...  MAD=0.19651  ρ=0.6800
  ε=0.500 ...  MAD=0.18906  ρ=0.6000
  ε=

In [4]:
"""
Causal SHAP (paper-faithful) for CSVs with auto regression/classification
- Tested for:
  /mnt/data/cv_risk.csv          (continuous target: cv_risk)
  /mnt/data/lung_cancer_risk.csv (continuous target: lung_cancer_risk)
"""

import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
from typing import Dict, List, Set, Optional
warnings.filterwarnings("ignore")

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import f1_score, accuracy_score, r2_score, mean_absolute_error
from sklearn.model_selection import train_test_split
from cgt_xai.causal_shap_robustness import run_robustness_analysis
from sklearn.utils.multiclass import type_of_target
import networkx as nx

try:
    from xgboost import XGBClassifier, XGBRegressor
    HAVE_XGB = True
except Exception:
    HAVE_XGB = False

HAVE_XGB = False

from causallearn.search.ConstraintBased.PC import pc

CONFIG = {
    'ALPHA_PC': 0.1,
    'MAX_PERM': 200,
    'M_RESAMPLES': 64,
    'GLOBAL_K': 30,
    'RANDOM_SEED': 42,
    'N_DAGS': 5
}
np.random.seed(CONFIG['RANDOM_SEED'])
rng = np.random.default_rng(CONFIG['RANDOM_SEED'])

COMMON_TARGET_NAMES = ["target","label","y","class","outcome",
                       "cvd","cardio","TenYearCHD","heart_disease","stroke","risk",
                       "cancer","lung_cancer","has_cancer","diagnosis"]

def detect_target_column(df: pd.DataFrame, user_hint: Optional[str]=None) -> str:
    if user_hint and user_hint in df.columns: return user_hint
    lower_map = {c.lower(): c for c in df.columns}
    for name in COMMON_TARGET_NAMES:
        if name.lower() in lower_map: return lower_map[name.lower()]
    return df.columns[-1]

def drop_unnamed(df: pd.DataFrame) -> pd.DataFrame:
    return df[[c for c in df.columns if not re.match(r'^unnamed', c, re.I)]].copy()

def to_numeric_df(df: pd.DataFrame) -> pd.DataFrame:
    num = df.select_dtypes(include=[np.number]).copy()
    for c in df.columns:
        if c not in num.columns:
            try:
                num[c] = pd.to_numeric(df[c], errors='raise')
            except Exception:
                pass
    return num

class CausalStructure:
    def __init__(self, alpha=0.05):
        self.alpha = alpha
        self.cpdag = None
        self.feature_names: List[str] = []
        self.dags: List[nx.DiGraph] = []

    def discover(self, X: pd.DataFrame):
        self.feature_names = list(X.columns)
        print(f"Running PC on {len(self.feature_names)} features...")
        res = pc(X.values, alpha=self.alpha, show_progress=False)
        self.cpdag = res.G.graph
        print("PC completed -> CPDAG")
        return self

    def get_skeleton(self) -> nx.DiGraph:
        G = nx.DiGraph(); G.add_nodes_from(self.feature_names)
        n = len(self.feature_names)
        for i in range(n):
            for j in range(n):
                if i == j: continue
                if self.cpdag[i, j] == -1 and self.cpdag[j, i] == 1:
                    G.add_edge(self.feature_names[i], self.feature_names[j])
                elif self.cpdag[i, j] == -1 and self.cpdag[j, i] == -1 and i < j:
                    G.add_edge(self.feature_names[i], self.feature_names[j])
                    G.add_edge(self.feature_names[j], self.feature_names[i])
        return G

    def sample_dags(self, n_samples: int) -> List[nx.DiGraph]:
        dags = []
        names = self.feature_names; n = len(names)
        undirected = []
        for i in range(n):
            for j in range(i+1, n):
                if self.cpdag[i, j] == -1 and self.cpdag[j, i] == -1:
                    undirected.append((names[i], names[j]))
        for _ in range(max(1, n_samples)):
            G = nx.DiGraph(); G.add_nodes_from(names)
            for i in range(n):
                for j in range(n):
                    if i != j and self.cpdag[i, j] == -1 and self.cpdag[j, i] == 1:
                        G.add_edge(names[i], names[j])
            for u, v in undirected:
                if rng.random() < 0.5: G.add_edge(u, v)
                else: G.add_edge(v, u)
                if not nx.is_directed_acyclic_graph(G):
                    if G.has_edge(u, v): G.remove_edge(u, v); G.add_edge(v, u)
                    else: G.remove_edge(v, u); G.add_edge(u, v)
                if not nx.is_directed_acyclic_graph(G):
                    if G.has_edge(u, v): G.remove_edge(u, v)
                    if G.has_edge(v, u): G.remove_edge(v, u)
            if not nx.is_directed_acyclic_graph(G):
                for (a, b) in list(G.edges()):
                    G.remove_edge(a, b)
                    if nx.is_directed_acyclic_graph(G): break
                    else: G.add_edge(a, b)
            dags.append(G)
        self.dags = dags
        print(f"Sampled {len(dags)} DAG(s)")
        return dags

class LinearMechanisms:
    def __init__(self, dag: nx.DiGraph, feature_names: List[str]):
        self.dag = dag; self.feature_names = feature_names
        self.mechanisms: Dict[str, Dict] = {}; self.topo: List[str] = []

    def fit(self, X: pd.DataFrame):
        try: self.topo = list(nx.topological_sort(self.dag))
        except Exception: self.topo = self._safe_topo()
        for var in self.feature_names:
            parents = list(self.dag.predecessors(var))
            if not parents:
                self.mechanisms[var] = {"parents": [], "intercept": float(X[var].mean()),
                                        "coef": None, "sigma": float(X[var].std(ddof=0))}
            else:
                lr = LinearRegression().fit(X[parents].values, X[var].values)
                resid = X[var].values - lr.predict(X[parents].values)
                self.mechanisms[var] = {"parents": parents, "intercept": float(lr.intercept_),
                                        "coef": lr.coef_.astype(float), "sigma": float(np.std(resid, ddof=0))}
        return self

    def _safe_topo(self) -> List[str]:
        ordered, remaining = [], set(self.feature_names)
        while remaining:
            ready = [n for n in remaining if all(p in ordered or p not in remaining
                                                 for p in self.dag.predecessors(n))]
            if not ready:
                ready = [min(remaining, key=lambda n: len([p for p in self.dag.predecessors(n) if p in remaining]))]
            for n in ready: ordered.append(n); remaining.remove(n)
        return ordered

    def sample_intervention(self, S: Set[str], x_obs: pd.Series) -> Dict[str, float]:
        x_new: Dict[str, float] = {}
        for var in self.topo:
            if var in S: x_new[var] = float(x_obs[var]); continue
            mech = self.mechanisms[var]
            if not mech["parents"]:
                mu, sigma = mech["intercept"], mech["sigma"]
                x_new[var] = mu if sigma < 1e-8 else float(rng.normal(mu, sigma))
            else:
                pvals = np.array([x_new.get(p, x_obs[p]) for p in mech["parents"]])
                mu = mech["intercept"] + pvals @ mech["coef"]
                sigma = mech["sigma"]
                x_new[var] = mu if sigma < 1e-8 else float(rng.normal(mu, sigma))
        return x_new

class CausalEffects:
    def __init__(self, dags: List[nx.DiGraph], feature_names: List[str]):
        self.dags = dags
        self.feature_names = feature_names
        self.effects: Dict[str, float] = {}

    def compute_effects(self, X: pd.DataFrame, y: pd.Series) -> Dict[str, float]:
        print("Computing IDA-style total effects W_i ...")
        y_vals = y.values
        totals = {f: [] for f in self.feature_names}

        for dag in self.dags:
            G = dag.copy()
            G.add_node("Y")

            # (A) Edge weights among features
            for u, v in dag.edges():
                parents = [p for p in dag.predecessors(v) if p != u]
                X_reg = np.column_stack([X[parents].values, X[u].values]) if parents else X[[u]].values
                lr = LinearRegression().fit(X_reg, X[v].values)
                w = float(abs(lr.coef_[-1]))
                if w > 1e-12:
                    G.add_edge(u, v, weight=w)

            # (B) Direct edges into Y: Y ~ all features
            paY = list(self.feature_names)
            lrY = LinearRegression().fit(X[paY].values, y_vals)
            for feat, w in zip(paY, np.abs(lrY.coef_)):
                if w > 1e-12:
                    G.add_edge(feat, "Y", weight=float(w))

            # (C) Sum of path products i -> ... -> Y
            for i_feat in self.feature_names:
                total = 0.0
                try:
                    for path in nx.all_simple_paths(G, i_feat, "Y", cutoff=10):
                        prod = 1.0
                        ok = True
                        for a, b in zip(path[:-1], path[1:]):
                            ed = G.get_edge_data(a, b)
                            if not ed or "weight" not in ed:
                                ok = False
                                break
                            prod *= ed["weight"]
                        if ok:
                            total += prod
                except Exception:
                    pass
                totals[i_feat].append(total)

        self.effects = {f: (float(np.mean(vals)) if len(vals) else 0.0) for f, vals in totals.items()}
        print("W_i computed")
        return self.effects

    def gamma_weights(self) -> Dict[str, float]:
        absW = {k: abs(v) for k, v in self.effects.items()}
        Z = sum(absW.values()) + 1e-12
        return {k: v / Z for k, v in absW.items()}


class CausalSHAP:
    def __init__(self, model, structure: CausalStructure, X_train: pd.DataFrame, y_train: pd.Series, task: str):
        self.model = model; self.structure = structure
        self.X_train = X_train; self.y_train = y_train
        self.features = list(X_train.columns); self.task = task

        print("\nFitting SEMs & sampling DAGs ...")
        self.dags = self.structure.sample_dags(CONFIG['N_DAGS'])
        self.sems = [LinearMechanisms(d, self.features).fit(X_train) for d in self.dags]

        self.effects = CausalEffects(self.dags, self.features)
        self.effects.compute_effects(X_train, y_train)
        self.gamma = self.effects.gamma_weights()
        self._Ef_cache = None
        print("CausalSHAP ready")

    def _predict_scalar(self, X_mat: np.ndarray) -> float:
        if self.task == "classification":
            proba = self.model.predict_proba(X_mat)[0]
            return float(proba[np.argmax(proba)])  # predicted class prob (paper used f(x) scalar)
        else:  # regression
            return float(self.model.predict(X_mat)[0])

    def _Ef(self) -> float:
        if self._Ef_cache is None:
            if self.task == "classification":
                P = self.model.predict_proba(self.X_train.values)
                self._Ef_cache = float(P.max(axis=1).mean())  # mean predicted-class prob
            else:
                preds = self.model.predict(self.X_train.values)
                self._Ef_cache = float(np.mean(preds))
        return self._Ef_cache

    def _vc(self, S: Set[str], x: pd.Series, M: int) -> float:
        total = 0.0
        per = max(1, M // max(1, len(self.sems)))
        for sem in self.sems:
            for _ in range(per):
                x_sample = sem.sample_intervention(S, x)
                X_mat = pd.DataFrame([x_sample], columns=self.features).values
                total += self._predict_scalar(X_mat)
        N = per * max(1, len(self.sems))
        return total / N if N > 0 else 0.0

    def explain_instance(self, x: pd.Series, n_perms: int=None, M: int=None) -> pd.Series:
        if n_perms is None: n_perms = CONFIG['MAX_PERM']
        if M is None: M = CONFIG['M_RESAMPLES']

        fx = self._predict_scalar(pd.DataFrame([x], columns=self.features).values)
        phi = np.zeros(len(self.features), dtype=float)

        for _ in range(n_perms):
            perm = rng.permutation(self.features)
            S: Set[str] = set()
            v_prev = self._vc(S, x, M)
            for feat in perm:
                S_next = S | {feat}
                v_curr = self._vc(S_next, x, M)
                delta = v_curr - v_prev
                phi[self.features.index(feat)] += self.gamma[feat] * delta
                v_prev, S = v_curr, S_next

        phi /= n_perms

        sum_phi_c = float(np.sum(phi))
        if abs(sum_phi_c) > 1e-12:
            Ef = self._Ef()
            phi = phi * ((fx - Ef) / sum_phi_c)
        return pd.Series(phi, index=self.features, name="Causal_SHAP")

    def explain_global(self, X_test: pd.DataFrame, k: int=None) -> pd.Series:
        if k is None: k = CONFIG['GLOBAL_K']
        k = min(k, len(X_test))
        idx = rng.choice(len(X_test), size=k, replace=False)
        print(f"\nComputing global Causal SHAP over {k} instances ...")
        exps = []
        for t, i in enumerate(idx, 1):
            if t % 5 == 0: print(f"  {t}/{k}")
            exps.append(self.explain_instance(X_test.iloc[i], n_perms=max(10, CONFIG['MAX_PERM']//4)))
        return pd.concat(exps, axis=1).mean(axis=1).rename("Causal_SHAP_Global")

def annotate_heatmap(ax, data: pd.DataFrame, title: str, cbar_label: str):
    im = ax.imshow(data.values, aspect='auto')
    ax.set_xticks(np.arange(data.shape[1])); ax.set_xticklabels(data.columns, rotation=45, ha='right')
    ax.set_yticks(np.arange(data.shape[0])); ax.set_yticklabels(data.index)
    for i in range(data.shape[0]):
        for j in range(data.shape[1]):
            ax.text(j, i, f"{data.values[i, j]:.3f}", ha='center', va='center', fontsize=8)
    cbar = plt.colorbar(im, ax=ax); cbar.set_label(cbar_label)
    ax.set_title(title)

def plot_comparison(df: pd.DataFrame, title="Feature Attribution Comparison"):
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    axes[0, 0].bar(np.arange(df.shape[0]), df.iloc[:, 0], width=0.6)
    axes[0, 0].set_title(f'{title}\n(Bar: first column)', fontsize=12, fontweight='bold')
    axes[0, 0].set_xticks(np.arange(df.shape[0])); axes[0, 0].set_xticklabels(df.index, rotation=45, ha='right')
    axes[0, 0].set_ylabel('Attribution'); axes[0, 0].grid(axis='y', alpha=0.3)
    annotate_heatmap(axes[0, 1], df.T, 'Attribution Heatmap', 'Score')
    df_norm = df.div(df.abs().max(axis=0), axis=1)
    for i, col in enumerate(df_norm.columns):
        axes[1, 0].bar(np.arange(df_norm.shape[0]) + (i-0.5)*(0.8/len(df_norm.columns)), df_norm[col],
                       width=0.8/len(df_norm.columns), label=col)
    axes[1, 0].set_title('Normalized by method max'); axes[1, 0].legend(fontsize=9); axes[1, 0].grid(axis='y', alpha=0.3)
    ranks = df.rank(ascending=False)
    annotate_heatmap(axes[1, 1], ranks.T, 'Ranking (lower is better)', 'Rank')
    fig.tight_layout(); return fig

def plot_causal_graph(G: nx.DiGraph, title="Discovered Causal Graph (PC)"):
    plt.figure(figsize=(8, 6))
    pos = nx.spring_layout(G, seed=CONFIG['RANDOM_SEED'], k=1.0, iterations=200)
    nx.draw_networkx_nodes(G, pos, node_color='lightblue', node_size=1200, alpha=0.9)
    nx.draw_networkx_edges(G, pos, edge_color='gray', arrows=True, arrowsize=15, width=2, arrowstyle='->', connectionstyle='arc3,rad=0.1')
    nx.draw_networkx_labels(G, pos, font_size=10, font_weight='bold')
    plt.title(title, fontsize=13, fontweight='bold', pad=12)
    plt.axis('off'); plt.tight_layout()
    return plt.gcf()

def run_causal_shap_on_csv(csv_path: str, dataset_name: Optional[str]=None, target_hint: Optional[str]=None, test_size: float=0.25):
    print("\n" + "="*72)
    print(f"Causal SHAP (paper-faithful) on: {csv_path}")
    print("="*72)

    try:
        raw = pd.read_csv(csv_path)
    except Exception as e:
        print(f"ERROR reading CSV: {e}"); return

    raw = drop_unnamed(raw)  # drop index columns like 'Unnamed: 0'
    if dataset_name is None:
        dataset_name = csv_path.split("/")[-1].split(".")[0]

    tgt_col = detect_target_column(raw, target_hint)
    if tgt_col not in raw.columns:
        print("ERROR: target column not found."); return

    y_raw = raw[tgt_col].copy()
    X_raw = raw.drop(columns=[tgt_col])

    X_num = to_numeric_df(X_raw)
    dropped = set(X_raw.columns) - set(X_num.columns)
    if dropped:
        print(f"Note: Dropping non-numeric columns: {sorted(list(dropped))}")

    X_num = X_num.replace([np.inf, -np.inf], np.nan).fillna(X_num.median(numeric_only=True))

    y_type = type_of_target(y_raw)
    if y_type in ["binary","multiclass"]:
        task = "classification"
        y_vec = y_raw.astype(int).values
    else:
        task = "regression"
        y_vec = pd.to_numeric(y_raw, errors="coerce").values
        if np.isnan(y_vec).any():
            print("ERROR: target has non-numeric values that cannot be coerced."); return

    strat = y_vec if (task=="classification" and len(np.unique(y_vec))>1) else None
    X_train, X_test, y_train, y_test = train_test_split(
        X_num, y_vec, test_size=test_size, random_state=CONFIG['RANDOM_SEED'], stratify=strat
    )

    scaler = StandardScaler()
    Xtr = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
    Xte = pd.DataFrame(scaler.transform(X_test),  columns=X_test.columns,  index=X_test.index)

    if task=="classification":
        n_classes = int(len(np.unique(y_train)))
        if HAVE_XGB:
            model = XGBClassifier(
                n_estimators=160, max_depth=5, learning_rate=0.08,
                subsample=0.9, colsample_bytree=0.9,
                random_state=CONFIG['RANDOM_SEED'], verbosity=0,
                objective="multi:softprob" if n_classes>2 else "binary:logistic",
                num_class=n_classes if n_classes>2 else None
            )
            print("Using XGBoost (classification)")
        else:
            model = RandomForestClassifier(n_estimators=160, max_depth=6, random_state=CONFIG['RANDOM_SEED'])
            print("Using RandomForest (classification)")
        model.fit(Xtr.values, y_train)
        y_pred = model.predict(Xte.values)
        acc = accuracy_score(y_test, y_pred); f1m = f1_score(y_test, y_pred, average='macro')
        print(f"Model: acc={acc:.4f}, macroF1={f1m:.4f}, classes={n_classes}")
    else:
        if HAVE_XGB:
            model = XGBRegressor(
                n_estimators=200, max_depth=5, learning_rate=0.08,
                subsample=0.9, colsample_bytree=0.9,
                random_state=CONFIG['RANDOM_SEED']
            )
            print("Using XGBoost (regression)")
        else:
            model = RandomForestRegressor(n_estimators=200, max_depth=6, random_state=CONFIG['RANDOM_SEED'])
            print("Using RandomForest (regression)")
        model.fit(Xtr.values, y_train)
        y_pred = model.predict(Xte.values)
        r2 = r2_score(y_test, y_pred); mae = mean_absolute_error(y_test, y_pred)
        print(f"Model: R2={r2:.4f}, MAE={mae:.4f}")

    # Step 1: PC discovery
    print("\n" + "-"*64)
    print("Step 1: Causal discovery (PC)")
    print("-"*64)
    cs = CausalStructure(alpha=CONFIG['ALPHA_PC']).discover(Xtr)
    G_skel = cs.get_skeleton()
    print(f"Graph: {G_skel.number_of_nodes()} nodes, {G_skel.number_of_edges()} edges")
    fig = plot_causal_graph(G_skel, title=f"Discovered Causal Graph (PC) — {dataset_name}")
    fig.savefig(f'causal_graph_{dataset_name}.png', dpi=150, bbox_inches='tight')
    print(f"Saved 'causal_graph_{dataset_name}.png'")

    # Step 2: Causal SHAP
    print("\n" + "-"*64)
    print("Step 2: Causal SHAP")
    print("-"*64)
    cshap = CausalSHAP(model, cs, Xtr, pd.Series(y_train), task=task)
    G_aug = make_augmented_graph_avg(cshap.dags, Xtr, pd.Series(y_train))
    fig_aug = plot_causal_graph_with_weights(G_aug, title=f"Augmented Causal Graph with target — {dataset_name}")
    fig_aug.savefig(f"causal_graph_with_target_{dataset_name}.png", dpi=150, bbox_inches="tight")
    print(f"Saved 'causal_graph_with_target_{dataset_name}.png'")
    print("\nTotal causal effects W_i and gamma_i:")
    for f in Xtr.columns:
        print(f"  {f:20s}  W={cshap.effects.effects[f]:10.6f}   γ={cshap.gamma[f]:.6f}")

    glob = cshap.explain_global(Xte, k=min(CONFIG['GLOBAL_K'], len(Xte)))

    # Step 3: Save/plot comparison (Causal_SHAP only by default for regression)
    cols = {"Causal_SHAP": glob}
    df = pd.DataFrame(cols)
    df.to_csv(f"attribution_scores_{dataset_name}.csv", index=True)
    print("\n" + "-"*64)
    print("Step 3: Comparison & Save")
    print("-"*64)
    print(df.round(6).to_string())
    print(f"\nSaved: attribution_scores_{dataset_name}.csv")

    fig2 = plot_comparison(df, title=f"Feature Attribution Comparison — {dataset_name}")
    fig2.savefig(f'attribution_comparison_{dataset_name}.png', dpi=150, bbox_inches='tight')
    print(f"Saved 'attribution_comparison_{dataset_name}.png'")

    try: plt.show()
    except Exception: pass


def make_augmented_graph_avg(dags: List[nx.DiGraph], X: pd.DataFrame, y: pd.Series) -> nx.DiGraph:
    """Average the feature→feature and feature→Y weights across sampled DAGs, then return a single augmented graph."""
    from collections import defaultdict

    edge_weights = defaultdict(list)
    nodes = set(X.columns.tolist() + ["Y"])

    for dag in dags:
        G = dag.copy()
        G.add_node("Y")

        # (A) feature->feature weights (parent-conditioned)
        for u, v in dag.edges():
            parents = [p for p in dag.predecessors(v) if p != u]
            X_reg = np.column_stack([X[parents].values, X[u].values]) if parents else X[[u]].values
            lr = LinearRegression().fit(X_reg, X[v].values)
            w = float(abs(lr.coef_[-1]))
            if w > 1e-12:
                edge_weights[(u, v)].append(w)

        # (B) feature->Y weights (Y ~ all features)
        lrY = LinearRegression().fit(X.values, y.values)
        for feat, w in zip(X.columns, np.abs(lrY.coef_)):
            if w > 1e-12:
                edge_weights[(feat, "Y")].append(float(w))

    G_avg = nx.DiGraph()
    G_avg.add_nodes_from(nodes)
    for (u, v), ws in edge_weights.items():
        G_avg.add_edge(u, v, weight=float(np.mean(ws)))
    return G_avg

def plot_causal_graph_with_weights(G: nx.DiGraph, title: str):
    import matplotlib.pyplot as plt
    pos = nx.spring_layout(G, seed=CONFIG['RANDOM_SEED'], k=1.0, iterations=200)
    plt.figure(figsize=(9, 7))
    nx.draw_networkx_nodes(G, pos, node_color='lightblue', node_size=1200, alpha=0.9)
    nx.draw_networkx_edges(G, pos, edge_color='gray', arrows=True, arrowsize=15,
                           width=2, arrowstyle='->', connectionstyle='arc3,rad=0.1')
    nx.draw_networkx_labels(G, pos, font_size=10, font_weight='bold')
    # edge labels = average weights
    edge_lbls = {(u, v): f"{d.get('weight', 0):.2f}" for u, v, d in G.edges(data=True)}
    nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_lbls, font_size=8)
    plt.title(title, fontsize=13, fontweight='bold', pad=12)
    plt.axis('off'); plt.tight_layout()
    return plt.gcf()

def make_augmented_graph_avg(dags: List[nx.DiGraph], X: pd.DataFrame, y: pd.Series) -> nx.DiGraph:
    """Average the feature→feature and feature→Y weights across sampled DAGs, then return a single augmented graph."""
    from collections import defaultdict

    edge_weights = defaultdict(list)
    nodes = set(X.columns.tolist() + ["Y"])

    for dag in dags:
        G = dag.copy()
        G.add_node("Y")

        # (A) feature->feature weights (parent-conditioned)
        for u, v in dag.edges():
            parents = [p for p in dag.predecessors(v) if p != u]
            X_reg = np.column_stack([X[parents].values, X[u].values]) if parents else X[[u]].values
            lr = LinearRegression().fit(X_reg, X[v].values)
            w = float(abs(lr.coef_[-1]))
            if w > 1e-12:
                edge_weights[(u, v)].append(w)

        # (B) feature->Y weights (Y ~ all features)
        lrY = LinearRegression().fit(X.values, y.values)
        for feat, w in zip(X.columns, np.abs(lrY.coef_)):
            if w > 1e-12:
                edge_weights[(feat, "Y")].append(float(w))

    G_avg = nx.DiGraph()
    G_avg.add_nodes_from(nodes)
    for (u, v), ws in edge_weights.items():
        G_avg.add_edge(u, v, weight=float(np.mean(ws)))
    return G_avg

def plot_causal_graph_with_weights(G: nx.DiGraph, title: str):
    import matplotlib.pyplot as plt
    pos = nx.spring_layout(G, seed=CONFIG['RANDOM_SEED'], k=1.0, iterations=200)
    plt.figure(figsize=(9, 7))
    nx.draw_networkx_nodes(G, pos, node_color='lightblue', node_size=1200, alpha=0.9)
    nx.draw_networkx_edges(G, pos, edge_color='gray', arrows=True, arrowsize=15,
                           width=2, arrowstyle='->', connectionstyle='arc3,rad=0.1')
    nx.draw_networkx_labels(G, pos, font_size=10, font_weight='bold')
    # edge labels = average weights
    edge_lbls = {(u, v): f"{d.get('weight', 0):.2f}" for u, v, d in G.edges(data=True)}
    nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_lbls, font_size=8)
    plt.title(title, fontsize=13, fontweight='bold', pad=12)
    plt.axis('off'); plt.tight_layout()
    return plt.gcf()

def main():
    run_causal_shap_on_csv("cv_risk.csv", dataset_name="cv_risk")
    run_causal_shap_on_csv("lung_cancer_risk.csv", dataset_name="lung_cancer_risk")

    run_robustness_analysis([
        {"path": "../../datasets/cv_risk.csv",         "name": "cv_risk"},
        {"path": "../../datasets/lung_cancer_risk.csv", "name": "lung_cancer_risk"},
    ])

if __name__ == "__main__":
    main()


Causal SHAP (paper-faithful) on: cv_risk.csv
ERROR reading CSV: [Errno 2] No such file or directory: 'cv_risk.csv'

Causal SHAP (paper-faithful) on: lung_cancer_risk.csv
ERROR reading CSV: [Errno 2] No such file or directory: 'lung_cancer_risk.csv'

  Robustness analysis: cv_risk  (datasets/cv_risk.csv)
Task: regression  |  4 features  |  750 train / 250 test samples
✓ Model trained
Running PC causal discovery ...
✓ Sampled 5 DAGs
✓ SEMs fitted
✓ γ weights computed

Computing baseline global SHAP (k=15) ...
Baseline φ₀:
  bmi                        -0.315155
  sleep_duration             -0.033617
  diet_score                 +0.029888
  family_history             +0.003239

Running robustness sweep over ε=[0.01, 0.05, 0.1, 0.2, 0.3, 0.5, 1.0] (5 trials each) ...
  ε=0.010 ...  MAD=0.23052  ρ=0.7600
  ε=0.050 ...  MAD=0.17774  ρ=0.6800
  ε=0.100 ...  MAD=0.20850  ρ=0.8800
  ε=0.200 ...  MAD=0.48671  ρ=0.9200
  ε=0.300 ...  MAD=0.19651  ρ=0.6800
  ε=0.500 ...  MAD=0.18906  ρ=0.6000
  ε=

In [ ]:
"""
causal_shap_robustness.py
=========================
Robustness analysis for Causal SHAP attributions under SEM weight perturbation.

PRECISE CLAIM (what this code tests and proves)
-----------------------------------------------
  "Causal SHAP attributions are sensitive to errors in SEM edge weight
   estimation, even when the causal graph structure is held fixed and correct.
   This experiment isolates SEM-parameter sensitivity from graph-structure
   uncertainty."

This is NOT a test of general Causal SHAP robustness.
It does NOT test: missing edges, wrong edge directions, or extra edges.

TWO EXPERIMENTAL ARMS
---------------------
  ARM 1 — Adversarial / worst-case perturbation
    Baseline weights W0 are fit on the full training set.
    For each epsilon in {0, 0.01, 0.02, 0.05, 0.10, 0.20, 0.30, 0.50, 1.0}:
      Perturb:  wij' = wij * (1 + epsilon * delta),   delta ~ Uniform(-1, 1)
      Repeat n_trials=20 times (different delta draws, same graph + instances).
      Measure:  MAD(phi_eps, phi_0)  and  Spearman rho of feature rankings.
    Find: eps* = first epsilon where noise-corrected MAD > 5% of |phi_0|.
    Interpretation: eps* is the weight-estimation error budget.

  ARM 2 — Estimation noise (realistic / constructive)
    The RF model and test instances are fixed throughout.
    SEM weights are refit on subsets of X_train of size n in {50,100,250,500,750}.
    Measure: MAD(phi_n, phi_full) as a function of n.
    Interpretation: shows how much training data stabilises attributions.

KEY VALIDITY CONTROLS
---------------------
  1. SHARED RANDOM SEEDS
     instance_indices, perm_seeds, noise_seeds generated ONCE.
     Reused for EVERY computation (baseline + all epsilon + all n).
     Only the SEM coefficients change between runs.
     MAD measures TRUE weight sensitivity, not MC noise.

  2. epsilon=0 NOISE FLOOR
     Running at epsilon=0 gives pure MC estimator noise.
     All reported MAD is noise-corrected: MAD_corr = max(0, MAD(eps) - MAD(0)).

  3. MONOTONICITY CHECK
     Noise-corrected MAD must increase with epsilon. Non-monotonicity means
     MC variance dominates — increase n_perms/M_per_sem before trusting results.

  4. CLASSIFICATION VALUE FUNCTION
     target_class locked to argmax(f(x)) of original x, held fixed across
     all coalition evaluations. Prevents moving-target sign-flip problem.

  5. GAMMA COMPUTATION
     - Feature->feature: structural OLS coefficients (parent-conditioned).
     - Feature->Y: LASSO selects pa(Y), then OLS on pa(Y) only.
       Avoids double-counting indirect paths.
     - Path products are SIGNED (not absolute) so cancellation is correct.
     - gamma_i = |total effect i->Y| / sum_j |total effect j->Y|

  6. WEAK-EDGE PROTECTION
     Edges with |wij| < 1e-3 are not perturbed (multiplicative perturbation
     of near-zero weights causes unrealistic sign flips).

OUTPUTS PER DATASET
-------------------
  arm1_adversarial_{name}.png     4-panel Arm 1 figure
  arm2_estimation_{name}.png      2-panel Arm 2 figure
  gamma_vs_sensitivity_{name}.png scatter: gamma_i vs sensitivity (mechanistic)
  arm1_scores_{name}.csv          numeric table per epsilon
  arm2_scores_{name}.csv          numeric table per n
  cross_dataset_noise_floor.png   cross-dataset comparison (if multiple)

USAGE
-----
  python causal_shap_robustness.py

  Or:
    from cgt_xai.causal_shap_robustness import run_all
    run_all([
        {"path": "cv_risk.csv",          "name": "cv_risk"},
        {"path": "lung_cancer_risk.csv", "name": "lung_cancer_risk"},
    ])
"""

import copy
import re
import warnings
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import networkx as nx

from scipy.stats import spearmanr
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.linear_model import LinearRegression, LassoCV
from sklearn.metrics import (r2_score, mean_absolute_error,
                             accuracy_score, f1_score)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.utils.multiclass import type_of_target
from typing import Dict, List, Optional, Set, Tuple

warnings.filterwarnings("ignore")
matplotlib.rcParams.update({"figure.dpi": 120})


# Configuration

CFG: dict = {
    "alpha_pc":          0.10,   # PC algorithm significance level
    "n_dags":            5,      # DAGs sampled from CPDAG
    "n_perms":           30,    # permutation samples per instance
    "M_per_sem":         16,     # interventional MC draws per SEM per coalition
    "k_instances":       5,     # test instances for global SHAP
    "epsilons":          [0.0, 0.05, 0.10, 0.30, 0.50, 1.0],
    "n_trials":          3,     # perturbation draws per epsilon
    "subsample_sizes":   [50, 100, 250, 500, 750],
    "breakdown_pct":     0.05,   # eps* threshold = 5% of |phi_0|
    "test_size":         0.25,
    "random_seed":       42,
    "parent_y_threshold": 1e-4,  # LASSO threshold for pa(Y) identification
}

COMMON_TARGETS = [
    "target", "label", "y", "class", "outcome", "cvd", "cardio",
    "TenYearCHD", "heart_disease", "stroke", "risk", "cancer",
    "lung_cancer", "has_cancer", "diagnosis", "cv_risk", "lung_cancer_risk",
]


# 1. data helpers

def _drop_unnamed(df: pd.DataFrame) -> pd.DataFrame:
    return df[[c for c in df.columns
               if not re.match(r"^unnamed", c, re.I)]].copy()


def _to_numeric(df: pd.DataFrame) -> pd.DataFrame:
    num = df.select_dtypes(include=[np.number]).copy()
    for c in df.columns:
        if c not in num.columns:
            try:
                num[c] = pd.to_numeric(df[c], errors="raise")
            except Exception:
                pass
    return num


def _detect_target(df: pd.DataFrame, hint: Optional[str] = None) -> str:
    if hint and hint in df.columns:
        return hint
    lmap = {c.lower(): c for c in df.columns}
    for t in COMMON_TARGETS:
        if t.lower() in lmap:
            return lmap[t.lower()]
    return df.columns[-1]


# 2. causal graph - pc + dag sampling

def _run_pc(X: pd.DataFrame, alpha: float) -> np.ndarray:
    """Run PC algorithm; return CPDAG adjacency matrix."""
    from causallearn.search.ConstraintBased.PC import pc as pc_alg
    result = pc_alg(X.values, alpha=alpha, show_progress=False)
    return result.G.graph


def _sample_dags(cpdag:    np.ndarray,
                 features: List[str],
                 n_dags:   int,
                 rng:      np.random.Generator) -> List[nx.DiGraph]:
    """
    Randomly orient undirected CPDAG edges into n_dags DAGs.
    Directed edges are copied exactly. Cycles are repaired by flipping then
    dropping the offending edge.
    """
    n = len(features)
    undirected = [
        (features[i], features[j])
        for i in range(n) for j in range(i + 1, n)
        if cpdag[i, j] == -1 and cpdag[j, i] == -1
    ]

    dags = []
    for _ in range(n_dags):
        G = nx.DiGraph()
        G.add_nodes_from(features)

        for i in range(n):
            for j in range(n):
                if i != j and cpdag[i, j] == -1 and cpdag[j, i] == 1:
                    G.add_edge(features[i], features[j])

        for u, v in undirected:
            G.add_edge(u, v) if rng.random() < 0.5 else G.add_edge(v, u)
            if not nx.is_directed_acyclic_graph(G):
                if G.has_edge(u, v): G.remove_edge(u, v); G.add_edge(v, u)
                else:                G.remove_edge(v, u); G.add_edge(u, v)
            if not nx.is_directed_acyclic_graph(G):
                for e in [(u, v), (v, u)]:
                    if G.has_edge(*e): G.remove_edge(*e)

        if not nx.is_directed_acyclic_graph(G):
            for a, b in list(G.edges()):
                G.remove_edge(a, b)
                if nx.is_directed_acyclic_graph(G): break
                G.add_edge(a, b)

        dags.append(G)
    return dags


# 3. sem - fit and perturb

class SEM:
    """
    Linear-Gaussian Structural Equation Model for one DAG.

    Xj = alpha_j + sum_{i in pa(j)} beta_ij * Xi + eps_j,   eps_j ~ N(0, sigma_j^2)

    Coefficients beta_ij are estimated by parent-conditioned OLS — the
    standard structural parameter estimator under linear-Gaussian SEMs.

    ARM 1 PERTURBATION:
        wij' = wij * (1 + epsilon * delta),   delta ~ Uniform(-1, 1)

    WEAK-EDGE PROTECTION:
        Edges where |beta_ij| < WEAK_EDGE_THRESHOLD are left unperturbed.
        Multiplicative perturbation of near-zero weights causes unrealistic
        sign flips (e.g. 0.001 -> -0.0008 at epsilon=1). Protecting them
        keeps the perturbed SEM physically interpretable.
    """

    WEAK_EDGE_THRESHOLD = 1e-3

    def __init__(self, dag: nx.DiGraph, features: List[str]):
        self.dag      = dag
        self.features = list(features)
        self.mechs:   Dict[str, dict] = {}
        self.topo:    List[str]       = []

    def fit(self,
            X:             pd.DataFrame,
            subsample_idx: Optional[np.ndarray] = None) -> "SEM":
        """
        Fit structural equations by parent-conditioned OLS.
        subsample_idx: if given, fit only on X.iloc[subsample_idx] (Arm 2).
        """
        Xf = X.iloc[subsample_idx] if subsample_idx is not None else X
        try:
            self.topo = list(nx.topological_sort(self.dag))
        except Exception:
            self.topo = self.features[:]

        for var in self.features:
            parents = list(self.dag.predecessors(var))
            if not parents:
                self.mechs[var] = {
                    "parents":   [],
                    "intercept": float(Xf[var].mean()),
                    "coef":      np.array([]),
                    "sigma":     float(max(Xf[var].std(ddof=0), 1e-9)),
                }
            else:
                lr    = LinearRegression().fit(Xf[parents].values, Xf[var].values)
                resid = Xf[var].values - lr.predict(Xf[parents].values)
                self.mechs[var] = {
                    "parents":   parents,
                    "intercept": float(lr.intercept_),
                    "coef":      lr.coef_.astype(float).copy(),
                    "sigma":     float(max(np.std(resid, ddof=0), 1e-9)),
                }
        return self

    def perturbed_mechs(self,
                        epsilon:  float,
                        pert_rng: np.random.Generator) -> Dict[str, dict]:
        """
        Deep-copy mechanisms with multiplicative weight perturbation.
        Weak edges (|coef| < WEAK_EDGE_THRESHOLD) are not perturbed.
        pert_rng is the ONLY random element that changes between runs.
        """
        mechs = {}
        for var, m in self.mechs.items():
            mc = copy.deepcopy(m)
            if mc["coef"].size > 0 and epsilon > 0.0:
                new_coef = mc["coef"].copy()
                for k, c in enumerate(mc["coef"]):
                    if abs(c) >= self.WEAK_EDGE_THRESHOLD:
                        delta       = pert_rng.uniform(-1.0, 1.0)
                        new_coef[k] = c * (1.0 + epsilon * delta)
                mc["coef"] = new_coef
            mechs[var] = mc
        return mechs

    def sample_intervention(self,
                            S:         Set[str],
                            x_obs:     pd.Series,
                            mechs:     Dict[str, dict],
                            noise_rng: np.random.Generator) -> Dict[str, float]:
        """
        Sample one interventional world do(X_S = x_S).
        noise_rng is passed in (shared, fixed) so baseline and perturbed
        runs draw identical Gaussian noise sequences.
        """
        x_new: Dict[str, float] = {}
        for var in self.topo:
            if var in S:
                x_new[var] = float(x_obs[var])
                continue
            m   = mechs[var]
            mu  = m["intercept"]
            sig = m["sigma"]
            if m["parents"]:
                pv  = np.array([x_new.get(p, float(x_obs[p])) for p in m["parents"]])
                mu += float(pv @ m["coef"])
            x_new[var] = mu if sig < 1e-9 else float(noise_rng.normal(mu, sig))
        return x_new


# 4. GAMMA WEIGHTS - corrected IDA-style proxy

def _identify_parents_of_Y(X:         pd.DataFrame,
                            y:         np.ndarray,
                            features:  List[str],
                            threshold: float) -> List[str]:
    """
    Identify direct structural parents of Y via LASSO selection.

    WHY NOT OLS ON ALL FEATURES?
    If X1->X2->Y (no direct X1->Y), OLS of Y~X1+X2 gives X1 a near-zero
    direct coefficient, but path X1->X2->Y is ALSO included in the path
    product sum. Using all-feature OLS for Y-weights double-counts the
    indirect contribution of X1.

    WHY LASSO?
    LASSO further drives indirect-effect features toward zero, giving a
    cleaner approximation of the direct-parent set pa(Y).
    After selection, we run OLS on pa(Y) only to get unbiased structural
    coefficients on those parents.

    LIMITATION: for very small samples LASSO may be unstable. We fall back
    to OLS with relative-magnitude thresholding in that case.
    """
    try:
        lasso = LassoCV(cv=min(5, len(y) // 10 + 1), max_iter=5000, random_state=0)
        lasso.fit(X[features].values, y)
        parents = [f for f, c in zip(features, lasso.coef_) if abs(c) > threshold]
    except Exception:
        lr      = LinearRegression().fit(X[features].values, y)
        max_c   = np.max(np.abs(lr.coef_)) + 1e-12
        parents = [f for f, c in zip(features, lr.coef_)
                   if abs(c) / max_c > threshold]

    if not parents:   # always keep at least the single strongest feature
        lr      = LinearRegression().fit(X[features].values, y)
        parents = [features[int(np.argmax(np.abs(lr.coef_)))]]

    return parents


def _compute_gamma(dags:      List[nx.DiGraph],
                   features:  List[str],
                   X:         pd.DataFrame,
                   y:         pd.Series,
                   sem_list:  List[SEM],
                   threshold: float = 1e-4,
                   verbose:   bool  = True) -> Dict[str, float]:
    """
    Compute normalised total causal effects gamma_i for each feature.

    ALGORITHM (per DAG):

    Step A — Feature->feature structural coefficients
      Read SIGNED coefficients from sem.mechs[var]["coef"].
      These are parent-conditioned OLS estimates = structural parameters.

    Step B — Feature->Y structural coefficients
      First identify pa(Y) via LASSO (see _identify_parents_of_Y).
      Then regress Y ~ pa(Y) ONLY (not all features) by OLS.
      Use SIGNED coefficients — preserves direction of causal effect.

      WHY THIS IS CORRECT:
      For feature Xi that is an INDIRECT parent of Y (Xi -> Xj -> Y),
      the direct OLS coefficient on Xi in Y~pa(Y) will be zero (Xi not in pa(Y)).
      Its effect on Y is accumulated in Step C via the path Xi->Xj->Y.
      No double-counting occurs.

    Step C — Total signed causal effect via path products
      For each feature i, enumerate all directed paths i->...->Y.
      For each path, multiply SIGNED structural coefficients along the path.
      Sum over all paths (cancellation between opposing paths is captured).

    Step D — Normalise
      gamma_i = |W_i| / sum_j |W_j|
      (absolute value taken AFTER path product accumulation;
       cancellation has already been correctly applied in Step C)

    NOTE: gamma is used as a magnitude weight in Causal SHAP (Heskes et al.),
    so the final normalised value is always non-negative. We state this
    explicitly in the thesis: gamma_i represents causal importance magnitude,
    not signed causal effect.
    """
    y_vals = y.values if hasattr(y, "values") else np.array(y)

    # Identify pa(Y) once from the full dataset
    parents_of_Y = _identify_parents_of_Y(X, y_vals, features, threshold)
    if verbose:
        print(f"  pa(Y) [LASSO-selected]: {parents_of_Y}")

    totals: Dict[str, List[float]] = {f: [] for f in features}

    for dag, sem in zip(dags, sem_list):

        # Step A: build augmented graph with SIGNED structural weights
        G = nx.DiGraph()
        G.add_nodes_from(features + ["Y"])

        for var in features:
            m = sem.mechs[var]
            for parent, coef in zip(m["parents"], m["coef"]):
                if abs(coef) > 1e-12:
                    G.add_edge(parent, var, weight=float(coef))   # Signed

        # Step B: Y-regression on pa(Y) only, SIGNED coefficients
        if parents_of_Y:
            lrY = LinearRegression().fit(
                X[parents_of_Y].values, y_vals
            )
            for feat, coef in zip(parents_of_Y, lrY.coef_):
                if abs(coef) > 1e-12:
                    G.add_edge(feat, "Y", weight=float(coef))      # Signed

        # Step C: sum of signed path products i -> ... -> Y
        for feat in features:
            total = 0.0
            try:
                for path in nx.all_simple_paths(
                    G, feat, "Y", cutoff=len(features) + 1
                ):
                    prod, ok = 1.0, True
                    for a, b in zip(path[:-1], path[1:]):
                        ed = G.get_edge_data(a, b)
                        if not ed or "weight" not in ed:
                            ok = False; break
                        prod *= ed["weight"]        # signed multiplication
                    if ok:
                        total += prod               # signed accumulation
            except nx.NetworkXError:
                pass
            totals[feat].append(total)

    # Step D: normalise
    effects: Dict[str, float] = {
        f: float(np.mean(v)) if v else 0.0
        for f, v in totals.items()
    }
    Z = sum(abs(v) for v in effects.values()) + 1e-12

    if verbose:
        print("  Signed total effects W_i (before normalisation):")
        for f, v in sorted(effects.items(), key=lambda x: -abs(x[1])):
            print(f"    {f:25s}  W={v:+.6f}   gamma={abs(v)/Z:.6f}")

    return {f: abs(v) / Z for f, v in effects.items()}


# 5. CAUSAL SHAP VALUE FUNCTION  v(S)

def _predict_scalar(model,
                    X_mat:        np.ndarray,
                    task:         str,
                    target_class: Optional[int] = None) -> float:
    """
    Scalar model output for the value function v(S).

    CLASSIFICATION: lock to target_class = argmax(f(x)) of the original
    instance x. Held fixed across ALL coalition evaluations to ensure
    v(S) is always measuring the same quantity (probability of the same class).
    Without this fix, coalition outputs can switch predicted class, making
    SHAP values incoherent.

    REGRESSION: return scalar prediction directly.
    """
    if task == "classification":
        proba = model.predict_proba(X_mat)[0]
        idx   = target_class if target_class is not None else int(np.argmax(proba))
        return float(proba[idx])
    return float(model.predict(X_mat)[0])


def _E_f(model, X_train: pd.DataFrame, task: str,
         target_class: Optional[int] = None) -> float:
    """Baseline E[f(X)] over the training distribution."""
    if task == "classification":
        P = model.predict_proba(X_train.values)
        if target_class is not None:
            return float(P[:, target_class].mean())
        return float(P.max(axis=1).mean())
    return float(model.predict(X_train.values).mean())


def _v_coalition(S:            Set[str],
                 x_obs:        pd.Series,
                 features:     List[str],
                 sems:         List[SEM],
                 mechs_list:   List[Dict],
                 model,
                 task:         str,
                 M_per_sem:    int,
                 noise_seeds:  List[int],
                 target_class: Optional[int] = None) -> float:
    """
    Estimate v(S) = E[f(X) | do(X_S = x_S)] by Monte Carlo.

    noise_seeds are shared across baseline and all perturbed runs so that
    Gaussian noise draws are IDENTICAL — only SEM coefficients change.
    """
    total, count = 0.0, 0
    seed_idx = 0
    for sem, mechs in zip(sems, mechs_list):
        for _ in range(M_per_sem):
            noise_rng = np.random.default_rng(
                noise_seeds[seed_idx % len(noise_seeds)]
            )
            seed_idx += 1
            xs = sem.sample_intervention(S, x_obs, mechs, noise_rng)

            X_mat = np.array([[xs[f] for f in features]], dtype=float)

            total += _predict_scalar(model, X_mat, task, target_class)
            count += 1
    return total / count if count > 0 else 0.0


# 6. causal shap - per instance and global

def _shap_instance(x:            pd.Series,
                   features:     List[str],
                   sems:         List[SEM],
                   mechs_list:   List[Dict],
                   gamma:        Dict[str, float],
                   model,
                   task:         str,
                   Ef:           float,
                   n_perms:      int,
                   M_per_sem:    int,
                   perm_seeds:   List[int],
                   noise_seeds:  List[int]) -> np.ndarray:
    """
    Causal SHAP for one instance (permutation estimator).

    perm_seeds and noise_seeds are identical across baseline and all perturbed
    runs — only mechs_list changes. This is what isolates weight sensitivity.

    EFFICIENCY RESCALING: phi is rescaled so sum(phi_i) = f(x) - E[f(X)].
    A large rescale scalar (> 20) warns that n_perms is too low.
    """
    X_mat_x = np.array([[x[f] for f in features]], dtype=float)
    target_class = None
    if task == "classification":
        target_class = int(np.argmax(model.predict_proba(X_mat_x)[0]))

    fx  = _predict_scalar(model, X_mat_x, task, target_class)
    phi = np.zeros(len(features))

    for perm_idx in range(n_perms):
        perm_rng = np.random.default_rng(perm_seeds[perm_idx])
        perm     = list(perm_rng.permutation(features))
        S:         Set[str] = set()
        v_prev   = _v_coalition(S, x, features, sems, mechs_list,
                                model, task, M_per_sem, noise_seeds, target_class)
        for feat in perm:
            S_next = S | {feat}
            v_curr = _v_coalition(S_next, x, features, sems, mechs_list,
                                  model, task, M_per_sem, noise_seeds, target_class)
            phi[features.index(feat)] += gamma[feat] * (v_curr - v_prev)
            v_prev, S = v_curr, S_next

    phi /= n_perms

    s = float(np.sum(phi))
    if abs(s) > 1e-12:
        scalar = (fx - Ef) / s
        if abs(scalar) > 20:
            print(f"    [WARN] rescale scalar={scalar:.1f} "
                  f"(sum_phi={s:.4f}, fx-Ef={fx-Ef:.4f}) "
                  f"— increase n_perms")
        phi = phi * scalar

    return phi


def _shap_global(X_test:           pd.DataFrame,
                 features:         List[str],
                 sems:             List[SEM],
                 mechs_list:       List[Dict],
                 gamma:            Dict[str, float],
                 model,
                 task:             str,
                 Ef:               float,
                 instance_indices: np.ndarray,
                 n_perms:          int,
                 M_per_sem:        int,
                 perm_seeds:       List[int],
                 noise_seeds:      List[int],
                 verbose:          bool = False) -> np.ndarray:
    """
    Global Causal SHAP: returns (k x n_features) matrix.
    All randomness controlled by fixed seeds — nothing leaks in.
    """
    phis = []
    for t, i in enumerate(instance_indices):
        if verbose:
            print(f"    instance {t+1}/{len(instance_indices)}", flush=True)
        phis.append(_shap_instance(
            X_test.iloc[i], features, sems, mechs_list,
            gamma, model, task, Ef,
            n_perms, M_per_sem, perm_seeds, noise_seeds,
        ))

    phi_mat = np.array(phis)

    if verbose:
        sign_pct = (phi_mat > 0).mean(axis=0)
        print("  Sign stability (% instances phi_i > 0):")
        for f, p in zip(features, sign_pct):
            flag = "  <- UNSTABLE" if 0.3 < p < 0.7 else ""
            print(f"    {f:25s}  {p:.2f}{flag}")

    return phi_mat


# 7. arm 1 - adversarial epsilon sweep

def _arm1_sweep(baseline_phi:     np.ndarray,
                X_test:           pd.DataFrame,
                features:         List[str],
                sems:             List[SEM],
                gamma:            Dict[str, float],
                model,
                task:             str,
                Ef:               float,
                instance_indices: np.ndarray,
                perm_seeds:       List[int],
                noise_seeds:      List[int],
                master_rng:       np.random.Generator,
                cfg:              dict) -> List[dict]:
    """
    For each epsilon, draw n_trials perturbation realisations.
    Each trial changes ONLY the SEM coefficients (via pert_rng).
    All other randomness is fixed. MAD measures weight sensitivity only.
    """
    phi0_mean = baseline_phi.mean(axis=0)
    rows      = []

    for eps in cfg["epsilons"]:
        print(f"  eps={eps:.3f} ...", end="", flush=True)
        trial_mads, trial_rhos, trial_feat_diffs = [], [], []

        for _ in range(cfg["n_trials"]):
            pert_rng        = np.random.default_rng(int(master_rng.integers(0, 2**32)))
            perturbed_mechs = [sem.perturbed_mechs(eps, pert_rng) for sem in sems]

            phi_eps      = _shap_global(
                X_test, features, sems, perturbed_mechs, gamma,
                model, task, Ef,
                instance_indices=instance_indices,
                n_perms=cfg["n_perms"], M_per_sem=cfg["M_per_sem"],
                perm_seeds=perm_seeds, noise_seeds=noise_seeds,
                verbose=False,
            )
            phi_eps_mean = phi_eps.mean(axis=0)
            diff         = np.abs(phi_eps_mean - phi0_mean)
            trial_mads.append(float(diff.mean()))
            trial_feat_diffs.append(diff)
            rho, _ = spearmanr(np.abs(phi0_mean), np.abs(phi_eps_mean))
            trial_rhos.append(float(rho) if not np.isnan(rho) else 1.0)

        rows.append({
            "epsilon":         eps,
            "mad_mean":        float(np.mean(trial_mads)),
            "mad_std":         float(np.std(trial_mads)),
            "per_feature_mad": np.stack(trial_feat_diffs).mean(axis=0),
            "spearman_mean":   float(np.mean(trial_rhos)),
            "spearman_std":    float(np.std(trial_rhos)),
        })
        r = rows[-1]
        print(f"  MAD={r['mad_mean']:.5f} +/- {r['mad_std']:.5f}"
              f"   rho={r['spearman_mean']:.4f} +/- {r['spearman_std']:.4f}")

    return rows


def _check_monotonicity(rows: List[dict]) -> None:
    noise_floor = next((r["mad_mean"] for r in rows if r["epsilon"] == 0.0), 0.0)
    nonzero     = [r for r in rows if r["epsilon"] > 0.0]
    corrected   = [max(0.0, r["mad_mean"] - noise_floor) for r in nonzero]
    eps_vals    = [r["epsilon"] for r in nonzero]
    violations  = [
        (eps_vals[i-1], eps_vals[i], corrected[i-1], corrected[i])
        for i in range(1, len(corrected))
        if corrected[i] < corrected[i-1] * 0.90
    ]
    if violations:
        print("\n  WARNING: MONOTONICITY VIOLATIONS (corrected MAD decreased):")
        for v in violations:
            print(f"    eps={v[0]:.3f}->{v[1]:.3f}: MAD {v[2]:.5f}->{v[3]:.5f}")
        print("  -> Increase n_perms / M_per_sem before trusting these results.\n")
    else:
        print("\n  OK: Monotonicity holds — results are reliable.\n")


# 8. arm 2 - estimation-noise sweep

def _arm2_sweep(baseline_phi:     np.ndarray,
                X_train:          pd.DataFrame,
                X_test:           pd.DataFrame,
                y_train:          pd.Series,
                features:         List[str],
                dags:             List[nx.DiGraph],
                model,
                task:             str,
                Ef:               float,
                instance_indices: np.ndarray,
                perm_seeds:       List[int],
                noise_seeds:      List[int],
                master_rng:       np.random.Generator,
                cfg:              dict) -> List[dict]:
    """
    For each subsample size n:
      - Draw a random subsample of X_train of size n.
      - Refit ALL SEM weights on that subsample only.
      - Recompute gamma from the subsample SEM + subsample data.
      - Compute global Causal SHAP with fixed seeds.
      - Measure MAD(phi_n, phi_full).

    RF model, graph, and test instances are FROZEN throughout.
    Subsample size is the sole variable.
    """
    phi0_mean = baseline_phi.mean(axis=0)
    n_full    = len(X_train)
    rows      = []

    for n in cfg["subsample_sizes"]:
        if n >= n_full:
            print(f"  n={n}: skipping (>= full training size {n_full})")
            continue
        print(f"  n={n} ...", end="", flush=True)

        sub_idx = master_rng.choice(n_full, size=n, replace=False)
        X_sub   = X_train.iloc[sub_idx].reset_index(drop=True)
        y_sub   = y_train.iloc[sub_idx].reset_index(drop=True)

        sems_n  = [SEM(dag, features).fit(X_train, subsample_idx=sub_idx)
                   for dag in dags]
        gamma_n = _compute_gamma(
            dags, features, X_sub, y_sub, sems_n,
            threshold=cfg["parent_y_threshold"], verbose=False,
        )

        phi_n        = _shap_global(
            X_test, features, sems_n, [s.mechs for s in sems_n], gamma_n,
            model, task, Ef,
            instance_indices=instance_indices,
            n_perms=cfg["n_perms"], M_per_sem=cfg["M_per_sem"],
            perm_seeds=perm_seeds, noise_seeds=noise_seeds,
            verbose=False,
        )
        phi_n_mean = phi_n.mean(axis=0)
        diff       = np.abs(phi_n_mean - phi0_mean)
        rho, _     = spearmanr(np.abs(phi0_mean), np.abs(phi_n_mean))

        rows.append({
            "n":               n,
            "mad_mean":        float(diff.mean()),
            "per_feature_mad": diff,
            "spearman":        float(rho) if not np.isnan(rho) else 1.0,
        })
        print(f"  MAD={diff.mean():.5f}  rho={rows[-1]['spearman']:.4f}")

    return rows


# 9. plots

def _plot_arm1(rows:      List[dict],
               features:  List[str],
               name:      str,
               phi0_mean: np.ndarray,
               cfg:       dict) -> Optional[float]:
    """4-panel Arm 1 figure. Returns eps*."""
    noise_floor = next((r["mad_mean"] for r in rows if r["epsilon"] == 0.0), 0.0)
    eps_vals    = [r["epsilon"]       for r in rows]
    mad_means   = [r["mad_mean"]      for r in rows]
    mad_stds    = [r["mad_std"]       for r in rows]
    corrected   = [max(0.0, m - noise_floor) for m in mad_means]
    rho_means   = [r["spearman_mean"] for r in rows]
    rho_stds    = [r["spearman_std"]  for r in rows]

    ref       = np.abs(phi0_mean).mean()
    threshold = cfg["breakdown_pct"] * ref if ref > 1e-10 else None
    eps_star  = None
    if threshold:
        for r, c in zip(rows, corrected):
            if r["epsilon"] == 0.0: continue
            if c > threshold:
                eps_star = r["epsilon"]; break

    fig, axes = plt.subplots(2, 2, figsize=(13, 9))
    fig.suptitle(
        f"ARM 1: SEM Weight Robustness — {name}\n"
        f"(causal graph fixed; only edge weights perturbed)",
        fontsize=12, fontweight="bold"
    )

    # raw MAD
    ax = axes[0, 0]
    lo = np.array(mad_means) - np.array(mad_stds)
    hi = np.array(mad_means) + np.array(mad_stds)
    ax.plot(eps_vals, mad_means, "o-", color="#185FA5", lw=2, ms=6,
            label="MAD mean (20 trials)")
    ax.fill_between(eps_vals, lo, hi, alpha=0.18, color="#185FA5", label="+/- 1 std")
    ax.axhline(noise_floor, color="#888", ls=":", lw=1.4,
               label=f"noise floor (eps=0): {noise_floor:.5f}")
    if threshold:
        ax.axhline(threshold, color="#A32D2D", ls="--", lw=1.3,
                   label=f"5% of |phi_0|: {threshold:.5f}")
    ax.set_xlabel("epsilon (relative weight perturbation)")
    ax.set_ylabel("MAD(phi_eps - phi_0)")
    ax.set_title("Raw MAD vs epsilon")
    ax.legend(fontsize=8); ax.grid(axis="y", alpha=0.3)

    # noise-corrected MAD
    ax = axes[0, 1]
    ax.plot(eps_vals, corrected, "s-", color="#0F6E56", lw=2, ms=6,
            label="MAD - noise floor")
    if threshold:
        ax.axhline(threshold, color="#A32D2D", ls="--", lw=1.3,
                   label=f"5% of |phi_0|: {threshold:.5f}")
    if eps_star:
        ax.axvline(eps_star, color="#A32D2D", ls=":", lw=1.5,
                   label=f"eps* = {eps_star:.2f}")
    ax.set_xlabel("epsilon (relative weight perturbation)")
    ax.set_ylabel("Noise-corrected MAD")
    ax.set_title(f"Corrected MAD vs epsilon   (eps* = {eps_star})")
    ax.legend(fontsize=8); ax.grid(axis="y", alpha=0.3)

    # per-feature heatmap
    ax = axes[1, 0]
    mat   = np.stack([r["per_feature_mad"] for r in rows], axis=1)
    order = np.argsort(-mat.sum(axis=1))
    mat_s = mat[order]
    fs    = [features[i] for i in order]
    im    = ax.imshow(mat_s, aspect="auto", cmap="YlOrRd")
    ax.set_xticks(np.arange(len(eps_vals)))
    ax.set_xticklabels([f"{e:.2f}" for e in eps_vals], rotation=30, ha="right", fontsize=8)
    ax.set_yticks(np.arange(len(fs))); ax.set_yticklabels(fs, fontsize=9)
    for i in range(mat_s.shape[0]):
        for j in range(mat_s.shape[1]):
            v = mat_s[i, j]
            ax.text(j, i, f"{v:.3f}", ha="center", va="center", fontsize=7,
                    color="white" if v > mat_s.max() * 0.6 else "black")
    plt.colorbar(im, ax=ax, label="|delta phi_i|")
    ax.set_title("Per-feature |delta phi_i| heatmap\n(sorted by total sensitivity)")

    # Spearman rho
    ax = axes[1, 1]
    lo = np.array(rho_means) - np.array(rho_stds)
    hi = np.array(rho_means) + np.array(rho_stds)
    ax.plot(eps_vals, rho_means, "D-", color="#0F6E56", lw=2, ms=6,
            label="Spearman rho mean")
    ax.fill_between(eps_vals, lo, hi, alpha=0.18, color="#0F6E56", label="+/- 1 std")
    ax.axhline(0.9, color="#854F0B", ls="--", lw=1.3, label="rho = 0.90")
    ax.set_ylim(-0.25, 1.08)
    ax.set_xlabel("epsilon (relative weight perturbation)")
    ax.set_ylabel("Spearman rho (rank correlation with phi_0)")
    ax.set_title("Feature-rank stability vs epsilon")
    ax.legend(fontsize=8); ax.grid(axis="y", alpha=0.3)

    fig.tight_layout()
    fname = f"arm1_adversarial_{name}.png"
    fig.savefig(fname, dpi=150, bbox_inches="tight")
    print(f"  Saved: {fname}")
    plt.close(fig)
    return eps_star


def _plot_arm2(rows:      List[dict],
               features:  List[str],
               name:      str,
               phi0_mean: np.ndarray) -> None:
    """2-panel Arm 2 figure."""
    if not rows:
        print("  Arm 2: no rows to plot.")
        return

    n_vals   = [r["n"]        for r in rows]
    mad_vals = [r["mad_mean"] for r in rows]
    rho_vals = [r["spearman"] for r in rows]

    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
    fig.suptitle(
        f"ARM 2: Estimation Noise (SEM subsample) — {name}\n"
        f"(RF model and graph fixed; only SEM training size varies)",
        fontsize=11, fontweight="bold"
    )

    axes[0].plot(n_vals, mad_vals, "o-", color="#185FA5", lw=2, ms=7)
    axes[0].set_xlabel("SEM training size n")
    axes[0].set_ylabel("MAD(phi_n - phi_full)")
    axes[0].set_title("Attribution drift vs SEM data size")
    axes[0].grid(alpha=0.3)
    axes[0].set_xticks(n_vals)
    axes[0].set_xticklabels([str(n) for n in n_vals])

    axes[1].plot(n_vals, rho_vals, "s-", color="#0F6E56", lw=2, ms=7)
    axes[1].axhline(0.9, color="#854F0B", ls="--", lw=1.3, label="rho = 0.90")
    axes[1].set_ylim(-0.25, 1.08)
    axes[1].set_xlabel("SEM training size n")
    axes[1].set_ylabel("Spearman rho (rank stability)")
    axes[1].set_title("Feature-rank stability vs SEM data size")
    axes[1].legend(fontsize=9); axes[1].grid(alpha=0.3)
    axes[1].set_xticks(n_vals)
    axes[1].set_xticklabels([str(n) for n in n_vals])

    fig.tight_layout()
    fname = f"arm2_estimation_{name}.png"
    fig.savefig(fname, dpi=150, bbox_inches="tight")
    print(f"  Saved: {fname}")
    plt.close(fig)


def _plot_gamma_sensitivity(arm1_rows: List[dict],
                            gamma:     Dict[str, float],
                            features:  List[str],
                            name:      str) -> None:
    """
    Scatter: gamma_i vs per-feature sensitivity at eps=1.0.
    A positive Spearman correlation here is the mechanistic validation:
    high-gamma features (strong causal path to Y) are most sensitive to
    weight perturbation, as the causal SHAP framework predicts.
    """
    max_row     = max((r for r in arm1_rows if r["epsilon"] > 0.0),
                      key=lambda r: r["epsilon"])
    sensitivity = max_row["per_feature_mad"]
    gamma_vals  = np.array([gamma[f] for f in features])

    corr, _ = spearmanr(gamma_vals, sensitivity)

    fig, ax = plt.subplots(figsize=(6, 5))
    ax.scatter(gamma_vals, sensitivity, s=100, color="#185FA5", zorder=3)
    for f, gv, sv in zip(features, gamma_vals, sensitivity):
        ax.annotate(f, (gv, sv), textcoords="offset points",
                    xytext=(6, 4), fontsize=9)
    if len(features) > 2:
        c  = np.polyfit(gamma_vals, sensitivity, 1)
        xs = np.linspace(gamma_vals.min(), gamma_vals.max(), 100)
        ax.plot(xs, np.polyval(c, xs), "--", color="#A32D2D", lw=1.5, label="OLS trend")
    ax.set_xlabel("gamma_i  (normalised total causal effect)", fontsize=11)
    ax.set_ylabel("|delta phi_i| at eps=1.0  (attribution shift)", fontsize=11)
    ax.set_title(
        f"Mechanistic check: gamma_i vs sensitivity — {name}\n"
        f"Spearman rho = {corr:.3f}  "
        f"(positive = high-gamma features most sensitive, as expected)",
        fontsize=10, fontweight="bold"
    )
    ax.legend(fontsize=9); ax.grid(alpha=0.3)
    fig.tight_layout()
    fname = f"gamma_vs_sensitivity_{name}.png"
    fig.savefig(fname, dpi=150, bbox_inches="tight")
    print(f"  Saved: {fname}")
    plt.close(fig)


def _plot_cross_dataset(all_results: List[dict]) -> None:
    """Stacked bar: noise floor vs true sensitivity per epsilon, per dataset."""
    n   = len(all_results)
    fig, axes = plt.subplots(1, n, figsize=(7 * n, 4.5), squeeze=False)
    for ax, res in zip(axes[0], all_results):
        rows      = res["arm1_rows"]
        nf        = res["noise_floor"]
        eps_vals  = [r["epsilon"]  for r in rows]
        corrected = [max(0.0, r["mad_mean"] - nf) for r in rows]
        x = np.arange(len(eps_vals))
        ax.bar(x, [nf] * len(eps_vals),
               color="salmon", alpha=0.85, label=f"noise floor ({nf:.4f})")
        ax.bar(x, corrected, bottom=[nf] * len(eps_vals),
               color="steelblue", alpha=0.85, label="true weight sensitivity")
        ax.set_xticks(x)
        ax.set_xticklabels([f"{e:.2f}" for e in eps_vals], rotation=30, fontsize=8)
        ax.set_xlabel("epsilon"); ax.set_ylabel("MAD")
        ax.set_title(f"{res['name']}\nnoise floor vs weight sensitivity")
        ax.legend(fontsize=9)
    fig.suptitle(
        "Cross-dataset noise floor decomposition\n"
        "(graph fixed; only SEM weights vary)",
        fontsize=11, fontweight="bold"
    )
    fig.tight_layout()
    fig.savefig("cross_dataset_noise_floor.png", dpi=150, bbox_inches="tight")
    print("Saved: cross_dataset_noise_floor.png")
    plt.close(fig)


# 10. main pipeline - one dataset

def _run_single(csv_path: str, name: str, cfg: dict) -> dict:
    """Full robustness pipeline for one dataset."""
    print(f"\n{'='*72}")
    print(f"  DATASET: {name}  ({csv_path})")
    print(f"{'='*72}")

    master_rng = np.random.default_rng(cfg["random_seed"])

    # load
    raw   = _drop_unnamed(pd.read_csv(csv_path))
    tgt   = _detect_target(raw)
    y_raw = raw[tgt]
    X_raw = _to_numeric(raw.drop(columns=[tgt]))
    X_raw = X_raw.replace([np.inf, -np.inf], np.nan).fillna(X_raw.median())

    y_type = type_of_target(y_raw)
    task   = "classification" if y_type in ["binary", "multiclass"] else "regression"
    y_vec  = (y_raw.astype(int).values if task == "classification"
              else pd.to_numeric(y_raw, errors="coerce").values)

    strat = y_vec if task == "classification" else None
    X_tr, X_te, y_tr, y_te = train_test_split(
        X_raw, y_vec,
        test_size=cfg["test_size"], random_state=cfg["random_seed"], stratify=strat,
    )
    scaler   = StandardScaler()
    Xtr      = pd.DataFrame(scaler.fit_transform(X_tr), columns=X_tr.columns)
    Xte      = pd.DataFrame(scaler.transform(X_te),     columns=X_te.columns)
    features = list(Xtr.columns)
    print(f"Task: {task}  |  {len(features)} features  |  "
          f"{len(Xtr)} train / {len(Xte)} test")

    # train RF (frozen for entire experiment)
    if task == "classification":
        model = RandomForestClassifier(n_estimators=200, max_depth=6,
                                       random_state=cfg["random_seed"])
        model.fit(Xtr.values, y_tr)
        y_pred = model.predict(Xte.values)
        print(f"RF: acc={accuracy_score(y_te, y_pred):.4f}  "
              f"macroF1={f1_score(y_te, y_pred, average='macro'):.4f}")
    else:
        model = RandomForestRegressor(n_estimators=200, max_depth=6,
                                      random_state=cfg["random_seed"])
        model.fit(Xtr.values, y_tr)
        y_pred = model.predict(Xte.values)
        print(f"RF: R2={r2_score(y_te, y_pred):.4f}  "
              f"MAE={mean_absolute_error(y_te, y_pred):.4f}")

    # causal discovery
    print("Running PC algorithm ...", flush=True)
    cpdag = _run_pc(Xtr, cfg["alpha_pc"])
    dags  = _sample_dags(cpdag, features, cfg["n_dags"], master_rng)
    print(f"OK: {len(dags)} DAGs sampled")

    # baseline SEMs on full training data
    sems_full = [SEM(dag, features).fit(Xtr) for dag in dags]
    print("OK: Baseline SEMs fitted")

    # gamma with corrected computation
    print("Computing gamma weights ...")
    gamma_full = _compute_gamma(
        dags, features, Xtr, pd.Series(y_tr),
        sems_full, threshold=cfg["parent_y_threshold"], verbose=True,
    )
    Ef = _E_f(model, Xtr, task)

    # pre-generate ALL shared seeds - generated ONCE, used everywhere
    k                = min(cfg["k_instances"], len(Xte))
    instance_indices = master_rng.choice(len(Xte), size=k, replace=False)
    perm_seeds       = [int(master_rng.integers(0, 2**32))
                        for _ in range(cfg["n_perms"])]
    noise_seeds      = [int(master_rng.integers(0, 2**32))
                        for _ in range(cfg["n_dags"] * cfg["M_per_sem"] * 20)]

    print(f"\nOK: Shared seeds locked")
    print(f"    {k} instances | {cfg['n_perms']} perm seeds | "
          f"{len(noise_seeds)} noise seeds")
    print(f"    (IDENTICAL for baseline + all epsilon + all n runs)\n")

    # baseline SHAP
    print(f"Computing baseline SHAP "
          f"(n_perms={cfg['n_perms']}, M_per_sem={cfg['M_per_sem']}, k={k}) ...")
    base_mechs   = [sem.mechs for sem in sems_full]
    baseline_phi = _shap_global(
        Xte, features, sems_full, base_mechs, gamma_full,
        model, task, Ef,
        instance_indices=instance_indices,
        n_perms=cfg["n_perms"], M_per_sem=cfg["M_per_sem"],
        perm_seeds=perm_seeds, noise_seeds=noise_seeds,
        verbose=True,
    )
    phi0_mean = baseline_phi.mean(axis=0)
    print("\nBaseline phi_0 (sorted by |phi|):")
    for f, v in sorted(zip(features, phi0_mean), key=lambda x: -abs(x[1])):
        print(f"  {f:25s}  {v:+.6f}")

    # ARM 1
    print(f"\n{'─'*60}")
    print(f"ARM 1: Adversarial epsilon sweep")
    print(f"  epsilon = {cfg['epsilons']}")
    print(f"  {cfg['n_trials']} trials per epsilon")
    print(f"  Graph FIXED | test instances FIXED | only SEM weights perturbed")
    print(f"{'─'*60}")
    arm1_rows = _arm1_sweep(
        baseline_phi=baseline_phi,
        X_test=Xte, features=features,
        sems=sems_full, gamma=gamma_full,
        model=model, task=task, Ef=Ef,
        instance_indices=instance_indices,
        perm_seeds=perm_seeds, noise_seeds=noise_seeds,
        master_rng=master_rng, cfg=cfg,
    )

    noise_floor = next((r["mad_mean"] for r in arm1_rows if r["epsilon"] == 0.0), 0.0)
    ref = np.abs(phi0_mean).mean()
    print(f"\n  MC noise floor (MAD at eps=0) : {noise_floor:.6f}")
    if ref > 1e-10:
        pct = 100 * noise_floor / ref
        print(f"  Noise floor as % of |phi_0|  : {pct:.2f}%")
        if pct > 5.0:
            print("  WARNING: noise floor > 5% — increase n_perms / M_per_sem.")
        else:
            print("  OK: noise floor negligible.")
    _check_monotonicity(arm1_rows)

    eps_star = _plot_arm1(arm1_rows, features, name, phi0_mean, cfg)
    print(f"  eps* = {eps_star}  "
          f"({'breakdown reached' if eps_star else 'stable across all epsilon'})")

    records = []
    for r in arm1_rows:
        rec = {
            "epsilon":       r["epsilon"],
            "mad_mean":      r["mad_mean"],
            "mad_std":       r["mad_std"],
            "mad_corrected": max(0.0, r["mad_mean"] - noise_floor),
            "spearman_mean": r["spearman_mean"],
            "spearman_std":  r["spearman_std"],
        }
        for i, f in enumerate(features):
            rec[f"mad_{f}"] = r["per_feature_mad"][i]
        records.append(rec)
    arm1_df = pd.DataFrame(records).set_index("epsilon")
    arm1_df.to_csv(f"arm1_scores_{name}.csv")
    print(f"  Saved: arm1_scores_{name}.csv")
    print(arm1_df[["mad_mean", "mad_corrected", "spearman_mean"]].round(6).to_string())

    # ARM 2
    print(f"\n{'─'*60}")
    print(f"ARM 2: Estimation-noise sweep")
    print(f"  n = {cfg['subsample_sizes']}")
    print(f"  RF + graph + test instances FIXED | only SEM weights vary with n")
    print(f"{'─'*60}")
    arm2_rows = _arm2_sweep(
        baseline_phi=baseline_phi,
        X_train=Xtr, X_test=Xte,
        y_train=pd.Series(y_tr),
        features=features, dags=dags,
        model=model, task=task, Ef=Ef,
        instance_indices=instance_indices,
        perm_seeds=perm_seeds, noise_seeds=noise_seeds,
        master_rng=master_rng, cfg=cfg,
    )
    _plot_arm2(arm2_rows, features, name, phi0_mean)

    records2 = []
    for r in arm2_rows:
        rec = {"n": r["n"], "mad": r["mad_mean"], "spearman": r["spearman"]}
        for i, f in enumerate(features):
            rec[f"mad_{f}"] = r["per_feature_mad"][i]
        records2.append(rec)
    arm2_df = pd.DataFrame(records2).set_index("n")
    arm2_df.to_csv(f"arm2_scores_{name}.csv")
    print(f"  Saved: arm2_scores_{name}.csv")
    print(arm2_df[["mad", "spearman"]].round(6).to_string())

    # mechanistic plot
    _plot_gamma_sensitivity(arm1_rows, gamma_full, features, name)

    return {
        "name":        name,
        "features":    features,
        "phi0_mean":   phi0_mean,
        "gamma":       gamma_full,
        "noise_floor": noise_floor,
        "eps_star":    eps_star,
        "arm1_rows":   arm1_rows,
        "arm2_rows":   arm2_rows,
    }


# 11. entry point

def run_all(datasets: List[dict], cfg: Optional[dict] = None) -> List[dict]:
    """
    Run full robustness analysis for one or more datasets.

    WHAT THIS PROVES:
        Causal SHAP attributions are sensitive to SEM edge weight estimation
        errors, even when the causal graph structure is held fixed and correct.
        eps* quantifies the weight estimation error budget.

    WHAT THIS DOES NOT PROVE:
        Robustness to graph errors (wrong edges, wrong orientations).
        That is a separate, complementary experiment.

    Parameters
    ----------
    datasets : list of {"path": str, "name": str}
    cfg      : optional overrides to CFG defaults
    """
    if cfg is None:
        cfg = CFG

    all_results = []
    for ds in datasets:
        result = _run_single(ds["path"], ds["name"], cfg)
        all_results.append(result)

    if len(all_results) > 1:
        _plot_cross_dataset(all_results)

    print(f"\n{'='*72}")
    print("  ROBUSTNESS ANALYSIS COMPLETE")
    print(f"{'='*72}")
    print("\n  Claim: SEM-weight robustness under FIXED graph structure.")
    print("  Scope: does NOT cover graph-structure errors.\n")

    for res in all_results:
        idx   = np.argmax(np.abs(res["phi0_mean"]))
        dom_f = res["features"][idx]
        dom_v = res["phi0_mean"][idx]
        print(f"  {res['name']}:")
        print(f"    Dominant feature : {dom_f}  phi_0={dom_v:+.4f}")
        print(f"    MC noise floor   : {res['noise_floor']:.6f}")
        if res["eps_star"]:
            print(f"    eps*             : {res['eps_star']:.2f}  "
                  f"(weights must be within {res['eps_star']*100:.0f}% relative error)")
        else:
            print(f"    eps*             : not reached — stable across all epsilon")
        print()

    return all_results


# Script entry point

if __name__ == "__main__":
    run_all([
        {"path": "../../datasets/lung_cancer_risk.csv", "name": "lung_cancer_risk"},
    ])


  DATASET: lung_cancer_risk  (datasets/lung_cancer_risk.csv)
Task: regression  |  3 features  |  750 train / 250 test
RF: R2=0.7086  MAE=1.5185
Running PC algorithm ...
OK: 5 DAGs sampled
OK: Baseline SEMs fitted
Computing gamma weights ...
  pa(Y) [LASSO-selected]: ['smoking', 'stress', 'drink_coffee']
  Signed total effects W_i (before normalisation):
    smoking                    W=+2.818595   gamma=0.588029
    stress                     W=+1.696499   gamma=0.353932
    drink_coffee               W=+0.278200   gamma=0.058039

OK: Shared seeds locked
    5 instances | 30 perm seeds | 1600 noise seeds
    (IDENTICAL for baseline + all epsilon + all n runs)

Computing baseline SHAP (n_perms=30, M_per_sem=16, k=5) ...
    instance 1/5
    instance 2/5
    instance 3/5
    instance 4/5
    instance 5/5
  Sign stability (% instances phi_i > 0):
    smoking                    0.80
    stress                     0.20
    drink_coffee               0.40  <- UNSTABLE

Baseline phi_0 (sorte

In [11]:
"""
causal_shap_robustness.py
=========================
Robustness analysis for Causal SHAP attributions under SEM weight perturbation.

PRECISE CLAIM (what this code tests and proves)
-----------------------------------------------
  "Causal SHAP attributions are sensitive to errors in SEM edge weight
   estimation, even when the causal graph structure is held fixed and correct.
   This experiment isolates SEM-parameter sensitivity from graph-structure
   uncertainty."

This is NOT a test of general Causal SHAP robustness.
It does NOT test: missing edges, wrong edge directions, or extra edges.

TWO EXPERIMENTAL ARMS
---------------------
  ARM 1 — Adversarial / worst-case perturbation
    Baseline weights W0 are fit on the full training set.
    For each epsilon in {0, 0.01, 0.02, 0.05, 0.10, 0.20, 0.30, 0.50, 1.0}:
      Perturb:  wij' = wij * (1 + epsilon * delta),   delta ~ Uniform(-1, 1)
      Repeat n_trials=20 times (different delta draws, same graph + instances).
      Measure:  MAD(phi_eps, phi_0)  and  Spearman rho of feature rankings.
    Find: eps* = first epsilon where noise-corrected MAD > 5% of |phi_0|.
    Interpretation: eps* is the weight-estimation error budget.

  ARM 2 — Estimation noise (realistic / constructive)
    The RF model and test instances are fixed throughout.
    SEM weights are refit on subsets of X_train of size n in {50,100,250,500,750}.
    Measure: MAD(phi_n, phi_full) as a function of n.
    Interpretation: shows how much training data stabilises attributions.

KEY VALIDITY CONTROLS
---------------------
  1. SHARED RANDOM SEEDS
     instance_indices, perm_seeds, noise_seeds generated ONCE.
     Reused for EVERY computation (baseline + all epsilon + all n).
     Only the SEM coefficients change between runs.
     MAD measures TRUE weight sensitivity, not MC noise.

  2. epsilon=0 NOISE FLOOR
     Running at epsilon=0 gives pure MC estimator noise.
     All reported MAD is noise-corrected: MAD_corr = max(0, MAD(eps) - MAD(0)).

  3. MONOTONICITY CHECK
     Noise-corrected MAD must increase with epsilon. Non-monotonicity means
     MC variance dominates — increase n_perms/M_per_sem before trusting results.

  4. CLASSIFICATION VALUE FUNCTION
     target_class locked to argmax(f(x)) of original x, held fixed across
     all coalition evaluations. Prevents moving-target sign-flip problem.

  5. GAMMA COMPUTATION
     - Feature->feature: structural OLS coefficients (parent-conditioned).
     - Feature->Y: LASSO selects pa(Y), then OLS on pa(Y) only.
       Avoids double-counting indirect paths.
     - Path products are SIGNED (not absolute) so cancellation is correct.
     - gamma_i = |total effect i->Y| / sum_j |total effect j->Y|

  6. WEAK-EDGE PROTECTION
     Edges with |wij| < 1e-3 are not perturbed (multiplicative perturbation
     of near-zero weights causes unrealistic sign flips).

OUTPUTS PER DATASET
-------------------
  arm1_adversarial_{name}.png     4-panel Arm 1 figure
  arm2_estimation_{name}.png      2-panel Arm 2 figure
  gamma_vs_sensitivity_{name}.png scatter: gamma_i vs sensitivity (mechanistic)
  arm1_scores_{name}.csv          numeric table per epsilon
  arm2_scores_{name}.csv          numeric table per n
  cross_dataset_noise_floor.png   cross-dataset comparison (if multiple)

USAGE
-----
  python causal_shap_robustness.py

  Or:
    from cgt_xai.causal_shap_robustness import run_all
    run_all([
        {"path": "cv_risk.csv",          "name": "cv_risk"},
        {"path": "lung_cancer_risk.csv", "name": "lung_cancer_risk"},
    ])
"""

import copy
import re
import warnings
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import networkx as nx

from scipy.stats import spearmanr
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.linear_model import LinearRegression, LassoCV
from sklearn.metrics import (r2_score, mean_absolute_error,
                             accuracy_score, f1_score)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.utils.multiclass import type_of_target
from typing import Dict, List, Optional, Set, Tuple

warnings.filterwarnings("ignore")
matplotlib.rcParams.update({"figure.dpi": 120})


# Configuration

CFG: dict = {
    "alpha_pc":          0.10,   # PC algorithm significance level
    "n_dags":            5,      # DAGs sampled from CPDAG
    "n_perms":           30,    # permutation samples per instance
    "M_per_sem":         16,     # interventional MC draws per SEM per coalition
    "k_instances":       5,     # test instances for global SHAP
    "epsilons":          [0.0, 0.05, 0.10, 0.30, 0.50, 1.0],
    "n_trials":          3,     # perturbation draws per epsilon
    "subsample_sizes":   [50, 100, 250, 500, 750],
    "breakdown_pct":     0.05,   # eps* threshold = 5% of |phi_0|
    "test_size":         0.25,
    "random_seed":       42,
    "parent_y_threshold": 1e-4,  # LASSO threshold for pa(Y) identification
}

COMMON_TARGETS = [
    "target", "label", "y", "class", "outcome", "cvd", "cardio",
    "TenYearCHD", "heart_disease", "stroke", "risk", "cancer",
    "lung_cancer", "has_cancer", "diagnosis", "cv_risk", "lung_cancer_risk",
]


# 1. data helpers

def _drop_unnamed(df: pd.DataFrame) -> pd.DataFrame:
    return df[[c for c in df.columns
               if not re.match(r"^unnamed", c, re.I)]].copy()


def _to_numeric(df: pd.DataFrame) -> pd.DataFrame:
    num = df.select_dtypes(include=[np.number]).copy()
    for c in df.columns:
        if c not in num.columns:
            try:
                num[c] = pd.to_numeric(df[c], errors="raise")
            except Exception:
                pass
    return num


def _detect_target(df: pd.DataFrame, hint: Optional[str] = None) -> str:
    if hint and hint in df.columns:
        return hint
    lmap = {c.lower(): c for c in df.columns}
    for t in COMMON_TARGETS:
        if t.lower() in lmap:
            return lmap[t.lower()]
    return df.columns[-1]


# 2. causal graph - pc + dag sampling

def _run_pc(X: pd.DataFrame, alpha: float) -> np.ndarray:
    """Run PC algorithm; return CPDAG adjacency matrix."""
    from causallearn.search.ConstraintBased.PC import pc as pc_alg
    result = pc_alg(X.values, alpha=alpha, show_progress=False)
    return result.G.graph


def _sample_dags(cpdag:    np.ndarray,
                 features: List[str],
                 n_dags:   int,
                 rng:      np.random.Generator) -> List[nx.DiGraph]:
    """
    Randomly orient undirected CPDAG edges into n_dags DAGs.
    Directed edges are copied exactly. Cycles are repaired by flipping then
    dropping the offending edge.
    """
    n = len(features)
    undirected = [
        (features[i], features[j])
        for i in range(n) for j in range(i + 1, n)
        if cpdag[i, j] == -1 and cpdag[j, i] == -1
    ]

    dags = []
    for _ in range(n_dags):
        G = nx.DiGraph()
        G.add_nodes_from(features)

        for i in range(n):
            for j in range(n):
                if i != j and cpdag[i, j] == -1 and cpdag[j, i] == 1:
                    G.add_edge(features[i], features[j])

        for u, v in undirected:
            G.add_edge(u, v) if rng.random() < 0.5 else G.add_edge(v, u)
            if not nx.is_directed_acyclic_graph(G):
                if G.has_edge(u, v): G.remove_edge(u, v); G.add_edge(v, u)
                else:                G.remove_edge(v, u); G.add_edge(u, v)
            if not nx.is_directed_acyclic_graph(G):
                for e in [(u, v), (v, u)]:
                    if G.has_edge(*e): G.remove_edge(*e)

        if not nx.is_directed_acyclic_graph(G):
            for a, b in list(G.edges()):
                G.remove_edge(a, b)
                if nx.is_directed_acyclic_graph(G): break
                G.add_edge(a, b)

        dags.append(G)
    return dags


# 3. sem - fit and perturb

class SEM:
    """
    Linear-Gaussian Structural Equation Model for one DAG.

    Xj = alpha_j + sum_{i in pa(j)} beta_ij * Xi + eps_j,   eps_j ~ N(0, sigma_j^2)

    Coefficients beta_ij are estimated by parent-conditioned OLS — the
    standard structural parameter estimator under linear-Gaussian SEMs.

    ARM 1 PERTURBATION:
        wij' = wij * (1 + epsilon * delta),   delta ~ Uniform(-1, 1)

    WEAK-EDGE PROTECTION:
        Edges where |beta_ij| < WEAK_EDGE_THRESHOLD are left unperturbed.
        Multiplicative perturbation of near-zero weights causes unrealistic
        sign flips (e.g. 0.001 -> -0.0008 at epsilon=1). Protecting them
        keeps the perturbed SEM physically interpretable.
    """

    WEAK_EDGE_THRESHOLD = 1e-3

    def __init__(self, dag: nx.DiGraph, features: List[str]):
        self.dag      = dag
        self.features = list(features)
        self.mechs:   Dict[str, dict] = {}
        self.topo:    List[str]       = []

    def fit(self,
            X:             pd.DataFrame,
            subsample_idx: Optional[np.ndarray] = None) -> "SEM":
        """
        Fit structural equations by parent-conditioned OLS.
        subsample_idx: if given, fit only on X.iloc[subsample_idx] (Arm 2).
        """
        Xf = X.iloc[subsample_idx] if subsample_idx is not None else X
        try:
            self.topo = list(nx.topological_sort(self.dag))
        except Exception:
            self.topo = self.features[:]

        for var in self.features:
            parents = list(self.dag.predecessors(var))
            if not parents:
                self.mechs[var] = {
                    "parents":   [],
                    "intercept": float(Xf[var].mean()),
                    "coef":      np.array([]),
                    "sigma":     float(max(Xf[var].std(ddof=0), 1e-9)),
                }
            else:
                lr    = LinearRegression().fit(Xf[parents].values, Xf[var].values)
                resid = Xf[var].values - lr.predict(Xf[parents].values)
                self.mechs[var] = {
                    "parents":   parents,
                    "intercept": float(lr.intercept_),
                    "coef":      lr.coef_.astype(float).copy(),
                    "sigma":     float(max(np.std(resid, ddof=0), 1e-9)),
                }
        return self

    def perturbed_mechs(self,
                        epsilon:  float,
                        pert_rng: np.random.Generator) -> Dict[str, dict]:
        """
        Deep-copy mechanisms with multiplicative weight perturbation.
        Weak edges (|coef| < WEAK_EDGE_THRESHOLD) are not perturbed.
        pert_rng is the ONLY random element that changes between runs.
        """
        mechs = {}
        for var, m in self.mechs.items():
            mc = copy.deepcopy(m)
            if mc["coef"].size > 0 and epsilon > 0.0:
                new_coef = mc["coef"].copy()
                for k, c in enumerate(mc["coef"]):
                    if abs(c) >= self.WEAK_EDGE_THRESHOLD:
                        delta       = pert_rng.uniform(-1.0, 1.0)
                        new_coef[k] = c * (1.0 + epsilon * delta)
                mc["coef"] = new_coef
            mechs[var] = mc
        return mechs

    def sample_intervention(self,
                            S:         Set[str],
                            x_obs:     pd.Series,
                            mechs:     Dict[str, dict],
                            noise_rng: np.random.Generator) -> Dict[str, float]:
        """
        Sample one interventional world do(X_S = x_S).
        noise_rng is passed in (shared, fixed) so baseline and perturbed
        runs draw identical Gaussian noise sequences.
        """
        x_new: Dict[str, float] = {}
        for var in self.topo:
            if var in S:
                x_new[var] = float(x_obs[var])
                continue
            m   = mechs[var]
            mu  = m["intercept"]
            sig = m["sigma"]
            if m["parents"]:
                pv  = np.array([x_new.get(p, float(x_obs[p])) for p in m["parents"]])
                mu += float(pv @ m["coef"])
            x_new[var] = mu if sig < 1e-9 else float(noise_rng.normal(mu, sig))
        return x_new


# 4. GAMMA WEIGHTS - corrected IDA-style proxy

def _identify_parents_of_Y(X:         pd.DataFrame,
                            y:         np.ndarray,
                            features:  List[str],
                            threshold: float) -> List[str]:
    """
    Identify direct structural parents of Y via LASSO selection.

    WHY NOT OLS ON ALL FEATURES?
    If X1->X2->Y (no direct X1->Y), OLS of Y~X1+X2 gives X1 a near-zero
    direct coefficient, but path X1->X2->Y is ALSO included in the path
    product sum. Using all-feature OLS for Y-weights double-counts the
    indirect contribution of X1.

    WHY LASSO?
    LASSO further drives indirect-effect features toward zero, giving a
    cleaner approximation of the direct-parent set pa(Y).
    After selection, we run OLS on pa(Y) only to get unbiased structural
    coefficients on those parents.

    LIMITATION: for very small samples LASSO may be unstable. We fall back
    to OLS with relative-magnitude thresholding in that case.
    """
    try:
        lasso = LassoCV(cv=min(5, len(y) // 10 + 1), max_iter=5000, random_state=0)
        lasso.fit(X[features].values, y)
        parents = [f for f, c in zip(features, lasso.coef_) if abs(c) > threshold]
    except Exception:
        lr      = LinearRegression().fit(X[features].values, y)
        max_c   = np.max(np.abs(lr.coef_)) + 1e-12
        parents = [f for f, c in zip(features, lr.coef_)
                   if abs(c) / max_c > threshold]

    if not parents:   # always keep at least the single strongest feature
        lr      = LinearRegression().fit(X[features].values, y)
        parents = [features[int(np.argmax(np.abs(lr.coef_)))]]

    return parents


def _compute_gamma(dags:      List[nx.DiGraph],
                   features:  List[str],
                   X:         pd.DataFrame,
                   y:         pd.Series,
                   sem_list:  List[SEM],
                   threshold: float = 1e-4,
                   verbose:   bool  = True) -> Dict[str, float]:
    """
    Compute normalised total causal effects gamma_i for each feature.

    ALGORITHM (per DAG):

    Step A — Feature->feature structural coefficients
      Read SIGNED coefficients from sem.mechs[var]["coef"].
      These are parent-conditioned OLS estimates = structural parameters.

    Step B — Feature->Y structural coefficients
      First identify pa(Y) via LASSO (see _identify_parents_of_Y).
      Then regress Y ~ pa(Y) ONLY (not all features) by OLS.
      Use SIGNED coefficients — preserves direction of causal effect.

      WHY THIS IS CORRECT:
      For feature Xi that is an INDIRECT parent of Y (Xi -> Xj -> Y),
      the direct OLS coefficient on Xi in Y~pa(Y) will be zero (Xi not in pa(Y)).
      Its effect on Y is accumulated in Step C via the path Xi->Xj->Y.
      No double-counting occurs.

    Step C — Total signed causal effect via path products
      For each feature i, enumerate all directed paths i->...->Y.
      For each path, multiply SIGNED structural coefficients along the path.
      Sum over all paths (cancellation between opposing paths is captured).

    Step D — Normalise
      gamma_i = |W_i| / sum_j |W_j|
      (absolute value taken AFTER path product accumulation;
       cancellation has already been correctly applied in Step C)

    NOTE: gamma is used as a magnitude weight in Causal SHAP (Heskes et al.),
    so the final normalised value is always non-negative. We state this
    explicitly in the thesis: gamma_i represents causal importance magnitude,
    not signed causal effect.
    """
    y_vals = y.values if hasattr(y, "values") else np.array(y)

    # Identify pa(Y) once from the full dataset
    parents_of_Y = _identify_parents_of_Y(X, y_vals, features, threshold)
    if verbose:
        print(f"  pa(Y) [LASSO-selected]: {parents_of_Y}")

    totals: Dict[str, List[float]] = {f: [] for f in features}

    for dag, sem in zip(dags, sem_list):

        # Step A: build augmented graph with SIGNED structural weights
        G = nx.DiGraph()
        G.add_nodes_from(features + ["Y"])

        for var in features:
            m = sem.mechs[var]
            for parent, coef in zip(m["parents"], m["coef"]):
                if abs(coef) > 1e-12:
                    G.add_edge(parent, var, weight=float(coef))   # Signed

        # Step B: Y-regression on pa(Y) only, SIGNED coefficients
        if parents_of_Y:
            lrY = LinearRegression().fit(
                X[parents_of_Y].values, y_vals
            )
            for feat, coef in zip(parents_of_Y, lrY.coef_):
                if abs(coef) > 1e-12:
                    G.add_edge(feat, "Y", weight=float(coef))      # Signed

        # Step C: sum of signed path products i -> ... -> Y
        for feat in features:
            total = 0.0
            try:
                for path in nx.all_simple_paths(
                    G, feat, "Y", cutoff=len(features) + 1
                ):
                    prod, ok = 1.0, True
                    for a, b in zip(path[:-1], path[1:]):
                        ed = G.get_edge_data(a, b)
                        if not ed or "weight" not in ed:
                            ok = False; break
                        prod *= ed["weight"]        # signed multiplication
                    if ok:
                        total += prod               # signed accumulation
            except nx.NetworkXError:
                pass
            totals[feat].append(total)

    # Step D: normalise
    effects: Dict[str, float] = {
        f: float(np.mean(v)) if v else 0.0
        for f, v in totals.items()
    }
    Z = sum(abs(v) for v in effects.values()) + 1e-12

    if verbose:
        print("  Signed total effects W_i (before normalisation):")
        for f, v in sorted(effects.items(), key=lambda x: -abs(x[1])):
            print(f"    {f:25s}  W={v:+.6f}   gamma={abs(v)/Z:.6f}")

    return {f: abs(v) / Z for f, v in effects.items()}


# 5. CAUSAL SHAP VALUE FUNCTION  v(S)

def _predict_scalar(model,
                    X_mat:        np.ndarray,
                    task:         str,
                    target_class: Optional[int] = None) -> float:
    """
    Scalar model output for the value function v(S).

    CLASSIFICATION: lock to target_class = argmax(f(x)) of the original
    instance x. Held fixed across ALL coalition evaluations to ensure
    v(S) is always measuring the same quantity (probability of the same class).
    Without this fix, coalition outputs can switch predicted class, making
    SHAP values incoherent.

    REGRESSION: return scalar prediction directly.
    """
    if task == "classification":
        proba = model.predict_proba(X_mat)[0]
        idx   = target_class if target_class is not None else int(np.argmax(proba))
        return float(proba[idx])
    return float(model.predict(X_mat)[0])


def _E_f(model, X_train: pd.DataFrame, task: str,
         target_class: Optional[int] = None) -> float:
    """Baseline E[f(X)] over the training distribution."""
    if task == "classification":
        P = model.predict_proba(X_train.values)
        if target_class is not None:
            return float(P[:, target_class].mean())
        return float(P.max(axis=1).mean())
    return float(model.predict(X_train.values).mean())


def _v_coalition(S:            Set[str],
                 x_obs:        pd.Series,
                 features:     List[str],
                 sems:         List[SEM],
                 mechs_list:   List[Dict],
                 model,
                 task:         str,
                 M_per_sem:    int,
                 noise_seeds:  List[int],
                 target_class: Optional[int] = None) -> float:
    """
    Estimate v(S) = E[f(X) | do(X_S = x_S)] by Monte Carlo.

    noise_seeds are shared across baseline and all perturbed runs so that
    Gaussian noise draws are IDENTICAL — only SEM coefficients change.
    """
    total, count = 0.0, 0
    seed_idx = 0
    for sem, mechs in zip(sems, mechs_list):
        for _ in range(M_per_sem):
            noise_rng = np.random.default_rng(
                noise_seeds[seed_idx % len(noise_seeds)]
            )
            seed_idx += 1
            xs = sem.sample_intervention(S, x_obs, mechs, noise_rng)

            X_mat = np.array([[xs[f] for f in features]], dtype=float)

            total += _predict_scalar(model, X_mat, task, target_class)
            count += 1
    return total / count if count > 0 else 0.0


# 6. causal shap - per instance and global

def _shap_instance(x:            pd.Series,
                   features:     List[str],
                   sems:         List[SEM],
                   mechs_list:   List[Dict],
                   gamma:        Dict[str, float],
                   model,
                   task:         str,
                   Ef:           float,
                   n_perms:      int,
                   M_per_sem:    int,
                   perm_seeds:   List[int],
                   noise_seeds:  List[int]) -> np.ndarray:
    """
    Causal SHAP for one instance (permutation estimator).

    perm_seeds and noise_seeds are identical across baseline and all perturbed
    runs — only mechs_list changes. This is what isolates weight sensitivity.

    EFFICIENCY RESCALING: phi is rescaled so sum(phi_i) = f(x) - E[f(X)].
    A large rescale scalar (> 20) warns that n_perms is too low.
    """
    X_mat_x = np.array([[x[f] for f in features]], dtype=float)
    target_class = None
    if task == "classification":
        target_class = int(np.argmax(model.predict_proba(X_mat_x)[0]))

    fx  = _predict_scalar(model, X_mat_x, task, target_class)
    phi = np.zeros(len(features))

    for perm_idx in range(n_perms):
        perm_rng = np.random.default_rng(perm_seeds[perm_idx])
        perm     = list(perm_rng.permutation(features))
        S:         Set[str] = set()
        v_prev   = _v_coalition(S, x, features, sems, mechs_list,
                                model, task, M_per_sem, noise_seeds, target_class)
        for feat in perm:
            S_next = S | {feat}
            v_curr = _v_coalition(S_next, x, features, sems, mechs_list,
                                  model, task, M_per_sem, noise_seeds, target_class)
            phi[features.index(feat)] += gamma[feat] * (v_curr - v_prev)
            v_prev, S = v_curr, S_next

    phi /= n_perms

    s = float(np.sum(phi))
    if abs(s) > 1e-12:
        scalar = (fx - Ef) / s
        if abs(scalar) > 20:
            print(f"    [WARN] rescale scalar={scalar:.1f} "
                  f"(sum_phi={s:.4f}, fx-Ef={fx-Ef:.4f}) "
                  f"— increase n_perms")
        phi = phi * scalar

    return phi


def _shap_global(X_test:           pd.DataFrame,
                 features:         List[str],
                 sems:             List[SEM],
                 mechs_list:       List[Dict],
                 gamma:            Dict[str, float],
                 model,
                 task:             str,
                 Ef:               float,
                 instance_indices: np.ndarray,
                 n_perms:          int,
                 M_per_sem:        int,
                 perm_seeds:       List[int],
                 noise_seeds:      List[int],
                 verbose:          bool = False) -> np.ndarray:
    """
    Global Causal SHAP: returns (k x n_features) matrix.
    All randomness controlled by fixed seeds — nothing leaks in.
    """
    phis = []
    for t, i in enumerate(instance_indices):
        if verbose:
            print(f"    instance {t+1}/{len(instance_indices)}", flush=True)
        phis.append(_shap_instance(
            X_test.iloc[i], features, sems, mechs_list,
            gamma, model, task, Ef,
            n_perms, M_per_sem, perm_seeds, noise_seeds,
        ))

    phi_mat = np.array(phis)

    if verbose:
        sign_pct = (phi_mat > 0).mean(axis=0)
        print("  Sign stability (% instances phi_i > 0):")
        for f, p in zip(features, sign_pct):
            flag = "  <- UNSTABLE" if 0.3 < p < 0.7 else ""
            print(f"    {f:25s}  {p:.2f}{flag}")

    return phi_mat


# 7. arm 1 - adversarial epsilon sweep

def _arm1_sweep(baseline_phi:     np.ndarray,
                X_test:           pd.DataFrame,
                features:         List[str],
                sems:             List[SEM],
                gamma:            Dict[str, float],
                model,
                task:             str,
                Ef:               float,
                instance_indices: np.ndarray,
                perm_seeds:       List[int],
                noise_seeds:      List[int],
                master_rng:       np.random.Generator,
                cfg:              dict) -> List[dict]:
    """
    For each epsilon, draw n_trials perturbation realisations.
    Each trial changes ONLY the SEM coefficients (via pert_rng).
    All other randomness is fixed. MAD measures weight sensitivity only.
    """
    phi0_mean = baseline_phi.mean(axis=0)
    rows      = []

    for eps in cfg["epsilons"]:
        print(f"  eps={eps:.3f} ...", end="", flush=True)
        trial_mads, trial_rhos, trial_feat_diffs = [], [], []

        for _ in range(cfg["n_trials"]):
            pert_rng        = np.random.default_rng(int(master_rng.integers(0, 2**32)))
            perturbed_mechs = [sem.perturbed_mechs(eps, pert_rng) for sem in sems]

            phi_eps      = _shap_global(
                X_test, features, sems, perturbed_mechs, gamma,
                model, task, Ef,
                instance_indices=instance_indices,
                n_perms=cfg["n_perms"], M_per_sem=cfg["M_per_sem"],
                perm_seeds=perm_seeds, noise_seeds=noise_seeds,
                verbose=False,
            )
            phi_eps_mean = phi_eps.mean(axis=0)
            diff         = np.abs(phi_eps_mean - phi0_mean)
            trial_mads.append(float(diff.mean()))
            trial_feat_diffs.append(diff)
            rho, _ = spearmanr(np.abs(phi0_mean), np.abs(phi_eps_mean))
            trial_rhos.append(float(rho) if not np.isnan(rho) else 1.0)

        rows.append({
            "epsilon":         eps,
            "mad_mean":        float(np.mean(trial_mads)),
            "mad_std":         float(np.std(trial_mads)),
            "per_feature_mad": np.stack(trial_feat_diffs).mean(axis=0),
            "spearman_mean":   float(np.mean(trial_rhos)),
            "spearman_std":    float(np.std(trial_rhos)),
        })
        r = rows[-1]
        print(f"  MAD={r['mad_mean']:.5f} +/- {r['mad_std']:.5f}"
              f"   rho={r['spearman_mean']:.4f} +/- {r['spearman_std']:.4f}")

    return rows


def _check_monotonicity(rows: List[dict]) -> None:
    noise_floor = next((r["mad_mean"] for r in rows if r["epsilon"] == 0.0), 0.0)
    nonzero     = [r for r in rows if r["epsilon"] > 0.0]
    corrected   = [max(0.0, r["mad_mean"] - noise_floor) for r in nonzero]
    eps_vals    = [r["epsilon"] for r in nonzero]
    violations  = [
        (eps_vals[i-1], eps_vals[i], corrected[i-1], corrected[i])
        for i in range(1, len(corrected))
        if corrected[i] < corrected[i-1] * 0.90
    ]
    if violations:
        print("\n  WARNING: MONOTONICITY VIOLATIONS (corrected MAD decreased):")
        for v in violations:
            print(f"    eps={v[0]:.3f}->{v[1]:.3f}: MAD {v[2]:.5f}->{v[3]:.5f}")
        print("  -> Increase n_perms / M_per_sem before trusting these results.\n")
    else:
        print("\n  OK: Monotonicity holds — results are reliable.\n")


# 8. arm 2 - estimation-noise sweep

def _arm2_sweep(baseline_phi:     np.ndarray,
                X_train:          pd.DataFrame,
                X_test:           pd.DataFrame,
                y_train:          pd.Series,
                features:         List[str],
                dags:             List[nx.DiGraph],
                model,
                task:             str,
                Ef:               float,
                instance_indices: np.ndarray,
                perm_seeds:       List[int],
                noise_seeds:      List[int],
                master_rng:       np.random.Generator,
                cfg:              dict) -> List[dict]:
    """
    For each subsample size n:
      - Draw a random subsample of X_train of size n.
      - Refit ALL SEM weights on that subsample only.
      - Recompute gamma from the subsample SEM + subsample data.
      - Compute global Causal SHAP with fixed seeds.
      - Measure MAD(phi_n, phi_full).

    RF model, graph, and test instances are FROZEN throughout.
    Subsample size is the sole variable.
    """
    phi0_mean = baseline_phi.mean(axis=0)
    n_full    = len(X_train)
    rows      = []

    for n in cfg["subsample_sizes"]:
        if n >= n_full:
            print(f"  n={n}: skipping (>= full training size {n_full})")
            continue
        print(f"  n={n} ...", end="", flush=True)

        sub_idx = master_rng.choice(n_full, size=n, replace=False)
        X_sub   = X_train.iloc[sub_idx].reset_index(drop=True)
        y_sub   = y_train.iloc[sub_idx].reset_index(drop=True)

        sems_n  = [SEM(dag, features).fit(X_train, subsample_idx=sub_idx)
                   for dag in dags]
        gamma_n = _compute_gamma(
            dags, features, X_sub, y_sub, sems_n,
            threshold=cfg["parent_y_threshold"], verbose=False,
        )

        phi_n        = _shap_global(
            X_test, features, sems_n, [s.mechs for s in sems_n], gamma_n,
            model, task, Ef,
            instance_indices=instance_indices,
            n_perms=cfg["n_perms"], M_per_sem=cfg["M_per_sem"],
            perm_seeds=perm_seeds, noise_seeds=noise_seeds,
            verbose=False,
        )
        phi_n_mean = phi_n.mean(axis=0)
        diff       = np.abs(phi_n_mean - phi0_mean)
        rho, _     = spearmanr(np.abs(phi0_mean), np.abs(phi_n_mean))

        rows.append({
            "n":               n,
            "mad_mean":        float(diff.mean()),
            "per_feature_mad": diff,
            "spearman":        float(rho) if not np.isnan(rho) else 1.0,
        })
        print(f"  MAD={diff.mean():.5f}  rho={rows[-1]['spearman']:.4f}")

    return rows


# 9. plots

def _plot_arm1(rows:      List[dict],
               features:  List[str],
               name:      str,
               phi0_mean: np.ndarray,
               cfg:       dict) -> Optional[float]:
    """4-panel Arm 1 figure. Returns eps*."""
    noise_floor = next((r["mad_mean"] for r in rows if r["epsilon"] == 0.0), 0.0)
    eps_vals    = [r["epsilon"]       for r in rows]
    mad_means   = [r["mad_mean"]      for r in rows]
    mad_stds    = [r["mad_std"]       for r in rows]
    corrected   = [max(0.0, m - noise_floor) for m in mad_means]
    rho_means   = [r["spearman_mean"] for r in rows]
    rho_stds    = [r["spearman_std"]  for r in rows]

    ref       = np.abs(phi0_mean).mean()
    threshold = cfg["breakdown_pct"] * ref if ref > 1e-10 else None
    eps_star  = None
    if threshold:
        for r, c in zip(rows, corrected):
            if r["epsilon"] == 0.0: continue
            if c > threshold:
                eps_star = r["epsilon"]; break

    fig, axes = plt.subplots(2, 2, figsize=(13, 9))
    fig.suptitle(
        f"ARM 1: SEM Weight Robustness — {name}\n"
        f"(causal graph fixed; only edge weights perturbed)",
        fontsize=12, fontweight="bold"
    )

    # raw MAD
    ax = axes[0, 0]
    lo = np.array(mad_means) - np.array(mad_stds)
    hi = np.array(mad_means) + np.array(mad_stds)
    ax.plot(eps_vals, mad_means, "o-", color="#185FA5", lw=2, ms=6,
            label="MAD mean (20 trials)")
    ax.fill_between(eps_vals, lo, hi, alpha=0.18, color="#185FA5", label="+/- 1 std")
    ax.axhline(noise_floor, color="#888", ls=":", lw=1.4,
               label=f"noise floor (eps=0): {noise_floor:.5f}")
    if threshold:
        ax.axhline(threshold, color="#A32D2D", ls="--", lw=1.3,
                   label=f"5% of |phi_0|: {threshold:.5f}")
    ax.set_xlabel("epsilon (relative weight perturbation)")
    ax.set_ylabel("MAD(phi_eps - phi_0)")
    ax.set_title("Raw MAD vs epsilon")
    ax.legend(fontsize=8); ax.grid(axis="y", alpha=0.3)

    # noise-corrected MAD
    ax = axes[0, 1]
    ax.plot(eps_vals, corrected, "s-", color="#0F6E56", lw=2, ms=6,
            label="MAD - noise floor")
    if threshold:
        ax.axhline(threshold, color="#A32D2D", ls="--", lw=1.3,
                   label=f"5% of |phi_0|: {threshold:.5f}")
    if eps_star:
        ax.axvline(eps_star, color="#A32D2D", ls=":", lw=1.5,
                   label=f"eps* = {eps_star:.2f}")
    ax.set_xlabel("epsilon (relative weight perturbation)")
    ax.set_ylabel("Noise-corrected MAD")
    ax.set_title(f"Corrected MAD vs epsilon   (eps* = {eps_star})")
    ax.legend(fontsize=8); ax.grid(axis="y", alpha=0.3)

    # per-feature heatmap
    ax = axes[1, 0]
    mat   = np.stack([r["per_feature_mad"] for r in rows], axis=1)
    order = np.argsort(-mat.sum(axis=1))
    mat_s = mat[order]
    fs    = [features[i] for i in order]
    im    = ax.imshow(mat_s, aspect="auto", cmap="YlOrRd")
    ax.set_xticks(np.arange(len(eps_vals)))
    ax.set_xticklabels([f"{e:.2f}" for e in eps_vals], rotation=30, ha="right", fontsize=8)
    ax.set_yticks(np.arange(len(fs))); ax.set_yticklabels(fs, fontsize=9)
    for i in range(mat_s.shape[0]):
        for j in range(mat_s.shape[1]):
            v = mat_s[i, j]
            ax.text(j, i, f"{v:.3f}", ha="center", va="center", fontsize=7,
                    color="white" if v > mat_s.max() * 0.6 else "black")
    plt.colorbar(im, ax=ax, label="|delta phi_i|")
    ax.set_title("Per-feature |delta phi_i| heatmap\n(sorted by total sensitivity)")

    # Spearman rho
    ax = axes[1, 1]
    lo = np.array(rho_means) - np.array(rho_stds)
    hi = np.array(rho_means) + np.array(rho_stds)
    ax.plot(eps_vals, rho_means, "D-", color="#0F6E56", lw=2, ms=6,
            label="Spearman rho mean")
    ax.fill_between(eps_vals, lo, hi, alpha=0.18, color="#0F6E56", label="+/- 1 std")
    ax.axhline(0.9, color="#854F0B", ls="--", lw=1.3, label="rho = 0.90")
    ax.set_ylim(-0.25, 1.08)
    ax.set_xlabel("epsilon (relative weight perturbation)")
    ax.set_ylabel("Spearman rho (rank correlation with phi_0)")
    ax.set_title("Feature-rank stability vs epsilon")
    ax.legend(fontsize=8); ax.grid(axis="y", alpha=0.3)

    fig.tight_layout()
    fname = f"arm1_adversarial_{name}.png"
    fig.savefig(fname, dpi=150, bbox_inches="tight")
    print(f"  Saved: {fname}")
    plt.close(fig)
    return eps_star


def _plot_arm2(rows:      List[dict],
               features:  List[str],
               name:      str,
               phi0_mean: np.ndarray) -> None:
    """2-panel Arm 2 figure."""
    if not rows:
        print("  Arm 2: no rows to plot.")
        return

    n_vals   = [r["n"]        for r in rows]
    mad_vals = [r["mad_mean"] for r in rows]
    rho_vals = [r["spearman"] for r in rows]

    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
    fig.suptitle(
        f"ARM 2: Estimation Noise (SEM subsample) — {name}\n"
        f"(RF model and graph fixed; only SEM training size varies)",
        fontsize=11, fontweight="bold"
    )

    axes[0].plot(n_vals, mad_vals, "o-", color="#185FA5", lw=2, ms=7)
    axes[0].set_xlabel("SEM training size n")
    axes[0].set_ylabel("MAD(phi_n - phi_full)")
    axes[0].set_title("Attribution drift vs SEM data size")
    axes[0].grid(alpha=0.3)
    axes[0].set_xticks(n_vals)
    axes[0].set_xticklabels([str(n) for n in n_vals])

    axes[1].plot(n_vals, rho_vals, "s-", color="#0F6E56", lw=2, ms=7)
    axes[1].axhline(0.9, color="#854F0B", ls="--", lw=1.3, label="rho = 0.90")
    axes[1].set_ylim(-0.25, 1.08)
    axes[1].set_xlabel("SEM training size n")
    axes[1].set_ylabel("Spearman rho (rank stability)")
    axes[1].set_title("Feature-rank stability vs SEM data size")
    axes[1].legend(fontsize=9); axes[1].grid(alpha=0.3)
    axes[1].set_xticks(n_vals)
    axes[1].set_xticklabels([str(n) for n in n_vals])

    fig.tight_layout()
    fname = f"arm2_estimation_{name}.png"
    fig.savefig(fname, dpi=150, bbox_inches="tight")
    print(f"  Saved: {fname}")
    plt.close(fig)


def _plot_gamma_sensitivity(arm1_rows: List[dict],
                            gamma:     Dict[str, float],
                            features:  List[str],
                            name:      str) -> None:
    """
    Scatter: gamma_i vs per-feature sensitivity at eps=1.0.
    A positive Spearman correlation here is the mechanistic validation:
    high-gamma features (strong causal path to Y) are most sensitive to
    weight perturbation, as the causal SHAP framework predicts.
    """
    max_row     = max((r for r in arm1_rows if r["epsilon"] > 0.0),
                      key=lambda r: r["epsilon"])
    sensitivity = max_row["per_feature_mad"]
    gamma_vals  = np.array([gamma[f] for f in features])

    corr, _ = spearmanr(gamma_vals, sensitivity)

    fig, ax = plt.subplots(figsize=(6, 5))
    ax.scatter(gamma_vals, sensitivity, s=100, color="#185FA5", zorder=3)
    for f, gv, sv in zip(features, gamma_vals, sensitivity):
        ax.annotate(f, (gv, sv), textcoords="offset points",
                    xytext=(6, 4), fontsize=9)
    if len(features) > 2:
        c  = np.polyfit(gamma_vals, sensitivity, 1)
        xs = np.linspace(gamma_vals.min(), gamma_vals.max(), 100)
        ax.plot(xs, np.polyval(c, xs), "--", color="#A32D2D", lw=1.5, label="OLS trend")
    ax.set_xlabel("gamma_i  (normalised total causal effect)", fontsize=11)
    ax.set_ylabel("|delta phi_i| at eps=1.0  (attribution shift)", fontsize=11)
    ax.set_title(
        f"Mechanistic check: gamma_i vs sensitivity — {name}\n"
        f"Spearman rho = {corr:.3f}  "
        f"(positive = high-gamma features most sensitive, as expected)",
        fontsize=10, fontweight="bold"
    )
    ax.legend(fontsize=9); ax.grid(alpha=0.3)
    fig.tight_layout()
    fname = f"gamma_vs_sensitivity_{name}.png"
    fig.savefig(fname, dpi=150, bbox_inches="tight")
    print(f"  Saved: {fname}")
    plt.close(fig)


def _plot_cross_dataset(all_results: List[dict]) -> None:
    """Stacked bar: noise floor vs true sensitivity per epsilon, per dataset."""
    n   = len(all_results)
    fig, axes = plt.subplots(1, n, figsize=(7 * n, 4.5), squeeze=False)
    for ax, res in zip(axes[0], all_results):
        rows      = res["arm1_rows"]
        nf        = res["noise_floor"]
        eps_vals  = [r["epsilon"]  for r in rows]
        corrected = [max(0.0, r["mad_mean"] - nf) for r in rows]
        x = np.arange(len(eps_vals))
        ax.bar(x, [nf] * len(eps_vals),
               color="salmon", alpha=0.85, label=f"noise floor ({nf:.4f})")
        ax.bar(x, corrected, bottom=[nf] * len(eps_vals),
               color="steelblue", alpha=0.85, label="true weight sensitivity")
        ax.set_xticks(x)
        ax.set_xticklabels([f"{e:.2f}" for e in eps_vals], rotation=30, fontsize=8)
        ax.set_xlabel("epsilon"); ax.set_ylabel("MAD")
        ax.set_title(f"{res['name']}\nnoise floor vs weight sensitivity")
        ax.legend(fontsize=9)
    fig.suptitle(
        "Cross-dataset noise floor decomposition\n"
        "(graph fixed; only SEM weights vary)",
        fontsize=11, fontweight="bold"
    )
    fig.tight_layout()
    fig.savefig("cross_dataset_noise_floor.png", dpi=150, bbox_inches="tight")
    print("Saved: cross_dataset_noise_floor.png")
    plt.close(fig)


# 10. main pipeline - one dataset

def _run_single(csv_path: str, name: str, cfg: dict) -> dict:
    """Full robustness pipeline for one dataset."""
    print(f"\n{'='*72}")
    print(f"  DATASET: {name}  ({csv_path})")
    print(f"{'='*72}")

    master_rng = np.random.default_rng(cfg["random_seed"])

    # load
    raw   = _drop_unnamed(pd.read_csv(csv_path))
    tgt   = _detect_target(raw)
    y_raw = raw[tgt]
    X_raw = _to_numeric(raw.drop(columns=[tgt]))
    X_raw = X_raw.replace([np.inf, -np.inf], np.nan).fillna(X_raw.median())

    y_type = type_of_target(y_raw)
    task   = "classification" if y_type in ["binary", "multiclass"] else "regression"
    y_vec  = (y_raw.astype(int).values if task == "classification"
              else pd.to_numeric(y_raw, errors="coerce").values)

    strat = y_vec if task == "classification" else None
    X_tr, X_te, y_tr, y_te = train_test_split(
        X_raw, y_vec,
        test_size=cfg["test_size"], random_state=cfg["random_seed"], stratify=strat,
    )
    scaler   = StandardScaler()
    Xtr      = pd.DataFrame(scaler.fit_transform(X_tr), columns=X_tr.columns)
    Xte      = pd.DataFrame(scaler.transform(X_te),     columns=X_te.columns)
    features = list(Xtr.columns)
    print(f"Task: {task}  |  {len(features)} features  |  "
          f"{len(Xtr)} train / {len(Xte)} test")

    # train RF (frozen for entire experiment)
    if task == "classification":
        model = RandomForestClassifier(n_estimators=200, max_depth=6,
                                       random_state=cfg["random_seed"])
        model.fit(Xtr.values, y_tr)
        y_pred = model.predict(Xte.values)
        print(f"RF: acc={accuracy_score(y_te, y_pred):.4f}  "
              f"macroF1={f1_score(y_te, y_pred, average='macro'):.4f}")
    else:
        model = RandomForestRegressor(n_estimators=200, max_depth=6,
                                      random_state=cfg["random_seed"])
        model.fit(Xtr.values, y_tr)
        y_pred = model.predict(Xte.values)
        print(f"RF: R2={r2_score(y_te, y_pred):.4f}  "
              f"MAE={mean_absolute_error(y_te, y_pred):.4f}")

    # causal discovery
    print("Running PC algorithm ...", flush=True)
    cpdag = _run_pc(Xtr, cfg["alpha_pc"])
    dags  = _sample_dags(cpdag, features, cfg["n_dags"], master_rng)
    print(f"OK: {len(dags)} DAGs sampled")

    # baseline SEMs on full training data
    sems_full = [SEM(dag, features).fit(Xtr) for dag in dags]
    print("OK: Baseline SEMs fitted")

    # gamma with corrected computation
    print("Computing gamma weights ...")
    gamma_full = _compute_gamma(
        dags, features, Xtr, pd.Series(y_tr),
        sems_full, threshold=cfg["parent_y_threshold"], verbose=True,
    )
    Ef = _E_f(model, Xtr, task)

    # pre-generate ALL shared seeds - generated ONCE, used everywhere
    k                = min(cfg["k_instances"], len(Xte))
    instance_indices = master_rng.choice(len(Xte), size=k, replace=False)
    perm_seeds       = [int(master_rng.integers(0, 2**32))
                        for _ in range(cfg["n_perms"])]
    noise_seeds      = [int(master_rng.integers(0, 2**32))
                        for _ in range(cfg["n_dags"] * cfg["M_per_sem"] * 20)]

    print(f"\nOK: Shared seeds locked")
    print(f"    {k} instances | {cfg['n_perms']} perm seeds | "
          f"{len(noise_seeds)} noise seeds")
    print(f"    (IDENTICAL for baseline + all epsilon + all n runs)\n")

    # baseline SHAP
    print(f"Computing baseline SHAP "
          f"(n_perms={cfg['n_perms']}, M_per_sem={cfg['M_per_sem']}, k={k}) ...")
    base_mechs   = [sem.mechs for sem in sems_full]
    baseline_phi = _shap_global(
        Xte, features, sems_full, base_mechs, gamma_full,
        model, task, Ef,
        instance_indices=instance_indices,
        n_perms=cfg["n_perms"], M_per_sem=cfg["M_per_sem"],
        perm_seeds=perm_seeds, noise_seeds=noise_seeds,
        verbose=True,
    )
    phi0_mean = baseline_phi.mean(axis=0)
    print("\nBaseline phi_0 (sorted by |phi|):")
    for f, v in sorted(zip(features, phi0_mean), key=lambda x: -abs(x[1])):
        print(f"  {f:25s}  {v:+.6f}")

    # ARM 1
    print(f"\n{'─'*60}")
    print(f"ARM 1: Adversarial epsilon sweep")
    print(f"  epsilon = {cfg['epsilons']}")
    print(f"  {cfg['n_trials']} trials per epsilon")
    print(f"  Graph FIXED | test instances FIXED | only SEM weights perturbed")
    print(f"{'─'*60}")
    arm1_rows = _arm1_sweep(
        baseline_phi=baseline_phi,
        X_test=Xte, features=features,
        sems=sems_full, gamma=gamma_full,
        model=model, task=task, Ef=Ef,
        instance_indices=instance_indices,
        perm_seeds=perm_seeds, noise_seeds=noise_seeds,
        master_rng=master_rng, cfg=cfg,
    )

    noise_floor = next((r["mad_mean"] for r in arm1_rows if r["epsilon"] == 0.0), 0.0)
    ref = np.abs(phi0_mean).mean()
    print(f"\n  MC noise floor (MAD at eps=0) : {noise_floor:.6f}")
    if ref > 1e-10:
        pct = 100 * noise_floor / ref
        print(f"  Noise floor as % of |phi_0|  : {pct:.2f}%")
        if pct > 5.0:
            print("  WARNING: noise floor > 5% — increase n_perms / M_per_sem.")
        else:
            print("  OK: noise floor negligible.")
    _check_monotonicity(arm1_rows)

    eps_star = _plot_arm1(arm1_rows, features, name, phi0_mean, cfg)
    print(f"  eps* = {eps_star}  "
          f"({'breakdown reached' if eps_star else 'stable across all epsilon'})")

    records = []
    for r in arm1_rows:
        rec = {
            "epsilon":       r["epsilon"],
            "mad_mean":      r["mad_mean"],
            "mad_std":       r["mad_std"],
            "mad_corrected": max(0.0, r["mad_mean"] - noise_floor),
            "spearman_mean": r["spearman_mean"],
            "spearman_std":  r["spearman_std"],
        }
        for i, f in enumerate(features):
            rec[f"mad_{f}"] = r["per_feature_mad"][i]
        records.append(rec)
    arm1_df = pd.DataFrame(records).set_index("epsilon")
    arm1_df.to_csv(f"arm1_scores_{name}.csv")
    print(f"  Saved: arm1_scores_{name}.csv")
    print(arm1_df[["mad_mean", "mad_corrected", "spearman_mean"]].round(6).to_string())

    # ARM 2
    print(f"\n{'─'*60}")
    print(f"ARM 2: Estimation-noise sweep")
    print(f"  n = {cfg['subsample_sizes']}")
    print(f"  RF + graph + test instances FIXED | only SEM weights vary with n")
    print(f"{'─'*60}")
    arm2_rows = _arm2_sweep(
        baseline_phi=baseline_phi,
        X_train=Xtr, X_test=Xte,
        y_train=pd.Series(y_tr),
        features=features, dags=dags,
        model=model, task=task, Ef=Ef,
        instance_indices=instance_indices,
        perm_seeds=perm_seeds, noise_seeds=noise_seeds,
        master_rng=master_rng, cfg=cfg,
    )
    _plot_arm2(arm2_rows, features, name, phi0_mean)

    records2 = []
    for r in arm2_rows:
        rec = {"n": r["n"], "mad": r["mad_mean"], "spearman": r["spearman"]}
        for i, f in enumerate(features):
            rec[f"mad_{f}"] = r["per_feature_mad"][i]
        records2.append(rec)
    arm2_df = pd.DataFrame(records2).set_index("n")
    arm2_df.to_csv(f"arm2_scores_{name}.csv")
    print(f"  Saved: arm2_scores_{name}.csv")
    print(arm2_df[["mad", "spearman"]].round(6).to_string())

    # mechanistic plot
    _plot_gamma_sensitivity(arm1_rows, gamma_full, features, name)

    return {
        "name":        name,
        "features":    features,
        "phi0_mean":   phi0_mean,
        "gamma":       gamma_full,
        "noise_floor": noise_floor,
        "eps_star":    eps_star,
        "arm1_rows":   arm1_rows,
        "arm2_rows":   arm2_rows,
    }


# 11. entry point

def run_all(datasets: List[dict], cfg: Optional[dict] = None) -> List[dict]:
    """
    Run full robustness analysis for one or more datasets.

    WHAT THIS PROVES:
        Causal SHAP attributions are sensitive to SEM edge weight estimation
        errors, even when the causal graph structure is held fixed and correct.
        eps* quantifies the weight estimation error budget.

    WHAT THIS DOES NOT PROVE:
        Robustness to graph errors (wrong edges, wrong orientations).
        That is a separate, complementary experiment.

    Parameters
    ----------
    datasets : list of {"path": str, "name": str}
    cfg      : optional overrides to CFG defaults
    """
    if cfg is None:
        cfg = CFG

    all_results = []
    for ds in datasets:
        result = _run_single(ds["path"], ds["name"], cfg)
        all_results.append(result)

    if len(all_results) > 1:
        _plot_cross_dataset(all_results)

    print(f"\n{'='*72}")
    print("  ROBUSTNESS ANALYSIS COMPLETE")
    print(f"{'='*72}")
    print("\n  Claim: SEM-weight robustness under FIXED graph structure.")
    print("  Scope: does NOT cover graph-structure errors.\n")

    for res in all_results:
        idx   = np.argmax(np.abs(res["phi0_mean"]))
        dom_f = res["features"][idx]
        dom_v = res["phi0_mean"][idx]
        print(f"  {res['name']}:")
        print(f"    Dominant feature : {dom_f}  phi_0={dom_v:+.4f}")
        print(f"    MC noise floor   : {res['noise_floor']:.6f}")
        if res["eps_star"]:
            print(f"    eps*             : {res['eps_star']:.2f}  "
                  f"(weights must be within {res['eps_star']*100:.0f}% relative error)")
        else:
            print(f"    eps*             : not reached — stable across all epsilon")
        print()

    return all_results


# Script entry point

if __name__ == "__main__":
    run_all([
        {"path": "../../datasets/cv_risk.csv", "name": "cv_risk"},
    ])


  DATASET: cv_risk  (datasets/cv_risk.csv)
Task: regression  |  4 features  |  750 train / 250 test
RF: R2=0.6623  MAE=1.3549
Running PC algorithm ...
OK: 5 DAGs sampled
OK: Baseline SEMs fitted
Computing gamma weights ...
  pa(Y) [LASSO-selected]: ['family_history', 'bmi']
  Signed total effects W_i (before normalisation):
    bmi                        W=+2.590158   gamma=0.457177
    diet_score                 W=+1.518097   gamma=0.267952
    sleep_duration             W=+1.459175   gamma=0.257552
    family_history             W=+0.098117   gamma=0.017318

OK: Shared seeds locked
    5 instances | 30 perm seeds | 1600 noise seeds
    (IDENTICAL for baseline + all epsilon + all n runs)

Computing baseline SHAP (n_perms=30, M_per_sem=16, k=5) ...
    instance 1/5
    instance 2/5
    instance 3/5
    instance 4/5
    instance 5/5
  Sign stability (% instances phi_i > 0):
    diet_score                 0.40  <- UNSTABLE
    sleep_duration             0.00
    family_history          